In [0]:
import pandas as pd
import pyspark
from pyspark.sql import functions as F

print("pandas : ", pd.__version__)
print("pyspark : ", pyspark.__version__)

pandas :  2.2.3
pyspark :  4.1.0


In [0]:
from math import exp

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.sql import functions as F
from pyspark.sql.functions import (add_months, avg, sum, col, count, countDistinct, floor, lead, lit, 
                                   months_between, percentile_approx, regexp_extract, row_number,
                                   max as spark_max, min as spark_min, 
                                   sum as spark_sum, abs as spark_abs,
                                   to_date, trim, upper, when
)

from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window



ANALYSIS_BASE_TABLE = "mortgage_risk.analytics.analysis_base_2024"
OUTPUT_TABLE_PREFIX = "mortgage_risk.analytics"


print("Analysis base table : ", ANALYSIS_BASE_TABLE)

Analysis base table :  mortgage_risk.analytics.analysis_base_2024


In [0]:
# Loading the curated mortgage dataset from Databricks and checking its structure
 
df = spark.table(ANALYSIS_BASE_TABLE)

print("Row count : ", df.count())
print("Column count : ", len(df.columns))
df.printSchema()

Row count :  14226973
Column count :  35
root
 |-- loan_sequence_number: string (nullable = true)
 |-- performance_quarter: string (nullable = true)
 |-- monthly_reporting_period: date (nullable = true)
 |-- current_actual_upb: double (nullable = true)
 |-- current_loan_delinquency_status: string (nullable = true)
 |-- loan_age: integer (nullable = true)
 |-- remaining_months_to_maturity: integer (nullable = true)
 |-- zero_balance_code: string (nullable = true)
 |-- zero_balance_effective_date: date (nullable = true)
 |-- current_interest_rate: double (nullable = true)
 |-- modification_flag: string (nullable = true)
 |-- estimated_loan_to_value: double (nullable = true)
 |-- delinquency_due_to_disaster: string (nullable = true)
 |-- borrower_assistance_status_code: string (nullable = true)
 |-- interest_bearing_upb: double (nullable = true)
 |-- origination_quarter: string (nullable = true)
 |-- first_payment_date: date (nullable = true)
 |-- credit_score: integer (nullable = true)
 

In [0]:
# Few sample rows
df.show(5, truncate = False)

+--------------------+-------------------+------------------------+------------------+-------------------------------+--------+----------------------------+-----------------+---------------------------+---------------------+-----------------+-----------------------+---------------------------+-------------------------------+--------------------+-------------------+------------------+------------+-------------------------+-----+----------------+----------------------+--------------------+------------+-------------+----------------------+-------+------------+--------------+-------------+------------+------------------+-------------------+---------------------+-----------------------+
|loan_sequence_number|performance_quarter|monthly_reporting_period|current_actual_upb|current_loan_delinquency_status|loan_age|remaining_months_to_maturity|zero_balance_code|zero_balance_effective_date|current_interest_rate|modification_flag|estimated_loan_to_value|delinquency_due_to_disaster|borrower_assist

In [0]:
# Columns and data types
pd.DataFrame({"column_name" : df.columns, 
              "data_type" : [dtype for _, dtype in df.dtypes]
    }
)

,column_name,data_type
0,loan_sequence_number,string
1,performance_quarter,string
2,monthly_reporting_period,date
3,current_actual_upb,double
4,current_loan_delinquency_status,string
5,loan_age,int
6,remaining_months_to_maturity,int
7,zero_balance_code,string
8,zero_balance_effective_date,date
9,current_interest_rate,double


In [0]:
# Checking important columns for missing values
key_columns = ["loan_sequence_number", "monthly_reporting_period", "current_loan_delinquency_status",
               "zero_balance_code", "credit_score", "property_state", "loan_purpose",
               "occupancy_status", "loan_to_value", "debt_to_income_ratio", "original_upb"
]

null_summary = df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
                          for c in key_columns
])

null_summary.show(truncate = False)

+--------------------+------------------------+-------------------------------+-----------------+------------+--------------+------------+----------------+-------------+--------------------+------------+
|loan_sequence_number|monthly_reporting_period|current_loan_delinquency_status|zero_balance_code|credit_score|property_state|loan_purpose|occupancy_status|loan_to_value|debt_to_income_ratio|original_upb|
+--------------------+------------------------+-------------------------------+-----------------+------------+--------------+------------+----------------+-------------+--------------------+------------+
|0                   |0                       |0                              |14131865         |0           |0             |0           |0               |0            |0                   |0           |
+--------------------+------------------------+-------------------------------+-----------------+------------+--------------+------------+----------------+-------------+---------------

In [0]:
# Summarizing numeric columns to understand their range, averages, distribution etc.
numeric_columns = ["credit_score", "loan_to_value", "combined_loan_to_value", "debt_to_income_ratio",
                   "original_upb", "current_actual_upb", "original_interest_rate", "current_interest_rate", "loan_age"
]

df.select(numeric_columns).summary().show(truncate = False)

+-------+------------------+------------------+----------------------+--------------------+------------------+------------------+----------------------+---------------------+-----------------+
|summary|credit_score      |loan_to_value     |combined_loan_to_value|debt_to_income_ratio|original_upb      |current_actual_upb|original_interest_rate|current_interest_rate|loan_age         |
+-------+------------------+------------------+----------------------+--------------------+------------------+------------------+----------------------+---------------------+-----------------+
|count  |14226973          |14226973          |14226973              |14226973            |14226973          |14226973          |14226973              |14226973             |14226973         |
|mean   |755.0521650670174 |75.01658490530627 |75.25713867595026     |37.88367420111081   |337095.63931835676|326144.18033829477|6.700121429521824     |6.700101537236937    |6.822888326279947|
|stddev |175.01117068970078|19.4737

In [0]:
# Cleaning key risk fields, standardizing delinquency codes and creating analysis flags such as delinquent, serious delinquent, prepaid and zero-balance

SERIOUS_SPECIAL_DELINQUENCY_STATUSES = ["RA"]

# 1. Standardising delinquency and zero-balance fields -> later logic uses consistent values
# 2. Validating core numeric borrower and loan fields -> unrealistic values are treated as missing
# 3. Translating raw delinquency codes into a usable numeric severity measure
# 4. Creating simple business flags : whether a loan is delinquent, seriously delinquent, prepaid or modified
analysis_df = (df.withColumn("delinquency_status_clean",
                             when(trim(col("current_loan_delinquency_status")) == "", None)
                             .otherwise(upper(trim(col("current_loan_delinquency_status"))))
                ).withColumn("zero_balance_code_clean", 
                             when(trim(col("zero_balance_code")) == "", None)
                             .otherwise(trim(col("zero_balance_code")))
                ).withColumn("credit_score_clean",
                             when((col("credit_score") >= 300) & (col("credit_score") <= 850), col("credit_score"))
                ).withColumn("loan_to_value_clean",
                             when((col("loan_to_value") >= 1) & (col("loan_to_value") <= 150), col("loan_to_value"))
                ).withColumn("combined_loan_to_value_clean",
                             when((col("combined_loan_to_value") >= 1) & (col("combined_loan_to_value") <= 150), col("combined_loan_to_value"))
                ).withColumn("debt_to_income_ratio_clean",
                             when((col("debt_to_income_ratio") >= 1) & (col("debt_to_income_ratio") <= 65), col("debt_to_income_ratio"))
                ).withColumn("delinquency_status_type",
                             when(col("delinquency_status_clean").isNull(), "missing_or_blank")
                             .when(regexp_extract(col("delinquency_status_clean"), r"^\d+$", 0) != "", "numeric")
                             .when(col("delinquency_status_clean").isin(SERIOUS_SPECIAL_DELINQUENCY_STATUSES), "special_serious")
                             .otherwise("unmapped_non_numeric")
                ).withColumn("delinquency_months", 
                             when(regexp_extract(col("delinquency_status_clean"), r"^\d+$", 0) != "", 
                                  regexp_extract(col("delinquency_status_clean"), r"^\d+$", 0).cast(IntegerType())
                            ).when(col("delinquency_status_clean").isin(SERIOUS_SPECIAL_DELINQUENCY_STATUSES),
                                   lit(999).cast(IntegerType())
                            )
                )
                .withColumn("is_delinquent", when(col("delinquency_months") >= 1, 1).otherwise(0)
                )
                .withColumn("is_serious_delinquent", when(col("delinquency_months") >= 3, 1).otherwise(0)
                )
                .withColumn("is_zero_balance", when(col("zero_balance_code_clean").isNotNull(), 1).otherwise(0)
                )
                .withColumn("is_prepay", when(col("zero_balance_code_clean") == "01", 1).otherwise(0)
                )
                .withColumn("is_modification", when(trim(col("modification_flag")) == "Y", 1).otherwise(0)
                )
)

# Previewing the cleaned fields and new flags to verify that the transformation logic
analysis_df.select("current_loan_delinquency_status", "delinquency_status_clean", "delinquency_status_type",
                   "delinquency_months", "zero_balance_code", "zero_balance_code_clean", "is_delinquent",
                   "is_serious_delinquent", "is_zero_balance", "is_prepay"
).show(20, truncate = False)

+-------------------------------+------------------------+-----------------------+------------------+-----------------+-----------------------+-------------+---------------------+---------------+---------+
|current_loan_delinquency_status|delinquency_status_clean|delinquency_status_type|delinquency_months|zero_balance_code|zero_balance_code_clean|is_delinquent|is_serious_delinquent|is_zero_balance|is_prepay|
+-------------------------------+------------------------+-----------------------+------------------+-----------------+-----------------------+-------------+---------------------+---------------+---------+
|0                              |0                       |numeric                |0                 |NULL             |NULL                   |0            |0                    |0              |0        |
|0                              |0                       |numeric                |0                 |NULL             |NULL                   |0            |0                  

In [0]:
# Rechecking missing values after cleaning to see how much usable data remains in the main risk fields
delinquency_status_audit = (analysis_df.groupBy("delinquency_status_clean", "delinquency_status_type")
                                    .agg(count("*").alias("row_count"),
                                         avg("is_delinquent").alias("delinquent_flag_rate"),
                                         avg("is_serious_delinquent").alias("serious_delinquent_flag_rate"),
                                         avg("is_zero_balance").alias("zero_balance_rate")
                                    )
                                    .orderBy(col("row_count").desc(), col("delinquency_status_clean"))
)

print("Delinquency status audit")
delinquency_status_audit.show(50, truncate = False)

Delinquency status audit
+------------------------+-----------------------+---------+--------------------+----------------------------+---------------------+
|delinquency_status_clean|delinquency_status_type|row_count|delinquent_flag_rate|serious_delinquent_flag_rate|zero_balance_rate    |
+------------------------+-----------------------+---------+--------------------+----------------------------+---------------------+
|0                       |numeric                |14103796 |0.0                 |0.0                         |0.006739887616071588 |
|1                       |numeric                |79055    |1.0                 |0.0                         |1.770918980456644E-4 |
|2                       |numeric                |17321    |1.0                 |0.0                         |1.1546677443565614E-4|
|3                       |numeric                |8302     |1.0                 |1.0                         |0.0                  |
|4                       |numeric           

In [0]:
analysis_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in [
        "credit_score_clean",
        "loan_to_value_clean",
        "combined_loan_to_value_clean",
        "debt_to_income_ratio_clean",
    ]
]).show(truncate = False)

+------------------+-------------------+----------------------------+--------------------------+
|credit_score_clean|loan_to_value_clean|combined_loan_to_value_clean|debt_to_income_ratio_clean|
+------------------+-------------------+----------------------------+--------------------------+
|4762              |79                 |88                          |168                       |
+------------------+-------------------+----------------------------+--------------------------+



In [0]:
# Creating business-friendly risk bands for credit score, LTV and DTI
# - Grouping continuous numeric values into interpretable borrower risk buckets
analysis_df = (analysis_df.withColumn("credit_score_band",
                                      when(col("credit_score_clean").isNull(), "Missing")
                                      .when(col("credit_score_clean") < 620, "<620")
                                      .when(col("credit_score_clean") < 680, "620-679")
                                      .when(col("credit_score_clean") < 740, "680-739")
                                      .when(col("credit_score_clean") < 800, "740-799")
                                      .otherwise("800+")
                        )
                        .withColumn("ltv_band",
                                    when(col("loan_to_value_clean").isNull(), "Missing")
                                    .when(col("loan_to_value_clean") < 60, "<60")
                                    .when(col("loan_to_value_clean") < 80, "60-79")
                                    .when(col("loan_to_value_clean") < 90, "80-89")
                                    .when(col("loan_to_value_clean") <= 100, "90-100")
                                    .otherwise("100+")
                        )
                        .withColumn("dti_band",
                                    when(col("debt_to_income_ratio_clean").isNull(), "Missing")
                                    .when(col("debt_to_income_ratio_clean") < 20, "<20")
                                    .when(col("debt_to_income_ratio_clean") < 30, "20-29")
                                    .when(col("debt_to_income_ratio_clean") < 40, "30-39")
                                    .when(col("debt_to_income_ratio_clean") < 50, "40-49")
                                    .otherwise("50+")
                        )
)

analysis_df.select("credit_score_clean", "credit_score_band", "loan_to_value_clean",
                   "ltv_band", "debt_to_income_ratio_clean", "dti_band"
).show(10, truncate = False)

+------------------+-----------------+-------------------+--------+--------------------------+--------+
|credit_score_clean|credit_score_band|loan_to_value_clean|ltv_band|debt_to_income_ratio_clean|dti_band|
+------------------+-----------------+-------------------+--------+--------------------------+--------+
|754               |740-799          |95.0               |90-100  |44.0                      |40-49   |
|754               |740-799          |95.0               |90-100  |44.0                      |40-49   |
|754               |740-799          |95.0               |90-100  |44.0                      |40-49   |
|754               |740-799          |95.0               |90-100  |44.0                      |40-49   |
|754               |740-799          |95.0               |90-100  |44.0                      |40-49   |
|754               |740-799          |95.0               |90-100  |44.0                      |40-49   |
|754               |740-799          |95.0               |90-100

In [0]:
# Keeping only the latest available record for each loan  (single current snapshot of loan status)
latest_window = Window.partitionBy("loan_sequence_number").orderBy(col("monthly_reporting_period").desc())

latest_loan_df = (analysis_df.repartition(8, "loan_sequence_number")
                            .withColumn("row_num", row_number().over(latest_window))
                            .filter(col("row_num") == 1)
                            .drop("row_num")
)

print("Latest snapshot row count : ", latest_loan_df.count())
print("Distinct loans in latest snapshot : ", latest_loan_df.select("loan_sequence_number").distinct().count())

# Ranking records within each loan from most recent to oldest
latest_loan_df.select("loan_sequence_number", "monthly_reporting_period", "delinquency_status_clean", 
                      "delinquency_months", "zero_balance_code_clean", "is_delinquent", "is_serious_delinquent",
                      "is_prepay"
).show(10, truncate = False)

Latest snapshot row count :  1048407
Distinct loans in latest snapshot :  1048407
+--------------------+------------------------+------------------------+------------------+-----------------------+-------------+---------------------+---------+
|loan_sequence_number|monthly_reporting_period|delinquency_status_clean|delinquency_months|zero_balance_code_clean|is_delinquent|is_serious_delinquent|is_prepay|
+--------------------+------------------------+------------------------+------------------+-----------------------+-------------+---------------------+---------+
|F24Q10000001        |2025-09-01              |0                       |0                 |NULL                   |0            |0                    |0        |
|F24Q10000026        |2025-09-01              |0                       |0                 |NULL                   |0            |0                    |0        |
|F24Q10000036        |2025-09-01              |0                       |0                 |NULL             

In [0]:
# Splitting the latest loan snapshot : active loans and terminated loans

active_latest_loan_df = (latest_loan_df.filter(col("is_zero_balance") == 0)
)

terminated_latest_loan_df = (latest_loan_df.filter(col("is_zero_balance") == 1)
)

print("Latest loans : ", latest_loan_df.count())
print("Active latest loans : ", active_latest_loan_df.count())
print("Terminated latest loans : ", terminated_latest_loan_df.count())

print("Latest snapshot month coverage : ")
latest_loan_df.groupBy("monthly_reporting_period").count().orderBy(col("monthly_reporting_period")
                                                                   .desc()).show(12, truncate = False)

print("Active latest snapshot month coverage : ")
active_latest_loan_df.groupBy("monthly_reporting_period").count().orderBy(col("monthly_reporting_period")
                                                                          .desc()).show(12, truncate = False)

Latest loans :  1048407
Active latest loans :  953299
Terminated latest loans :  95108
Latest snapshot month coverage : 
+------------------------+------+
|monthly_reporting_period|count |
+------------------------+------+
|2025-09-01              |964574|
|2025-08-01              |7985  |
|2025-07-01              |7945  |
|2025-06-01              |6898  |
|2025-05-01              |7822  |
|2025-04-01              |9213  |
|2025-03-01              |6948  |
|2025-02-01              |4517  |
|2025-01-01              |4773  |
|2024-12-01              |4625  |
|2024-11-01              |4990  |
|2024-10-01              |7162  |
+------------------------+------+
only showing top 12 rows
Active latest snapshot month coverage : 
+------------------------+------+
|monthly_reporting_period|count |
+------------------------+------+
|2025-09-01              |953299|
+------------------------+------+



In [0]:
# Capturing each loan's month-6 outcome
# - Choosing the best available record at or before month 6 to compare early loan behavior consistently

month_6_status_window = (Window.partitionBy("loan_sequence_number")
                                .orderBy(col("loan_age").desc(), col("monthly_reporting_period").desc())
)

month_6_status_df = (analysis_df.filter((col("loan_age") <= 6) & ((col("loan_age") == 6) | (col("is_zero_balance") == 1)))
                                .withColumn("row_num", row_number().over(month_6_status_window))
                                .filter(col("row_num") == 1)
                                .drop("row_num")
)

print("Month 6 status row count : ", month_6_status_df.count())
print("Distinct loans in month 6 status : ", month_6_status_df.select("loan_sequence_number").distinct().count())

# Prioritizing the latest valid observation within the first 6 months of loan life
month_6_status_df.select("loan_sequence_number", "loan_age", "monthly_reporting_period", "is_zero_balance",
                         "is_prepay", "loan_purpose", "credit_score_band"
).show(10, truncate = False)

Month 6 status row count :  1047937
Distinct loans in month 6 status :  1047937
+--------------------+--------+------------------------+---------------+---------+------------+-----------------+
|loan_sequence_number|loan_age|monthly_reporting_period|is_zero_balance|is_prepay|loan_purpose|credit_score_band|
+--------------------+--------+------------------------+---------------+---------+------------+-----------------+
|F24Q10000006        |6       |2024-08-01              |0              |0        |P           |800+             |
|F24Q10000044        |6       |2024-08-01              |0              |0        |P           |680-739          |
|F24Q10000050        |6       |2024-08-01              |0              |0        |P           |680-739          |
|F24Q10000054        |6       |2024-08-01              |0              |0        |P           |740-799          |
|F24Q10000060        |6       |2024-08-01              |0              |0        |P           |620-679          |
|F24Q100

In [0]:
# Summarizing overall portfolio and active portfolio
# - measuring top-level termination, prepayment, modification and delinquency rates.
overall_portfolio_summary = latest_loan_df.agg(count("*").alias("total_loans"),
                                               countDistinct("loan_sequence_number").alias("distinct_loans"),
                                               avg("is_zero_balance").alias("share_terminated_by_latest_observation"),
                                               avg("is_prepay").alias("share_prepaid_by_latest_observation"),
                                               avg("is_modification").alias("share_modified_by_latest_observation")
)

active_portfolio_summary = active_latest_loan_df.agg(count("*").alias("active_loans"),
                                                     countDistinct("loan_sequence_number").alias("distinct_active_loans"),
                                                     avg("is_delinquent").alias("active_delinquency_rate"),
                                                     avg("is_serious_delinquent").alias("active_serious_delinquency_rate")
)

overall_portfolio_summary.show(truncate = False)
active_portfolio_summary.show(truncate = False)

+-----------+--------------+--------------------------------------+-----------------------------------+------------------------------------+
|total_loans|distinct_loans|share_terminated_by_latest_observation|share_prepaid_by_latest_observation|share_modified_by_latest_observation|
+-----------+--------------+--------------------------------------+-----------------------------------+------------------------------------+
|1048407    |1048407       |0.0907166777787634                    |0.08759479858490071                |9.633663262454371E-5                |
+-----------+--------------+--------------------------------------+-----------------------------------+------------------------------------+

+------------+---------------------+-----------------------+-------------------------------+
|active_loans|distinct_active_loans|active_delinquency_rate|active_serious_delinquency_rate|
+------------+---------------------+-----------------------+-------------------------------+
|953299      |9

In [0]:
# Measuring active portfolio exposure in dollar terms
# - Quantifying how much unpaid principal is tied to delinquent and serious-delinquent loans.

# Converting delinquent UPB into portfolio exposure shares
active_exposure_summary = active_latest_loan_df.agg(count("*").alias("active_loan_count"),
                                                    spark_sum("current_actual_upb").alias("active_total_upb"),
                                                    spark_sum(when(col("is_delinquent") == 1, col("current_actual_upb"))
                                                              .otherwise(0)).alias("delinquent_upb"),
                                                    spark_sum(when(col("is_serious_delinquent") == 1, 
                                                                   col("current_actual_upb"))
                                                              .otherwise(0)).alias("serious_delinquent_upb")
).withColumn("delinquent_upb_share", col("delinquent_upb") / col("active_total_upb")
     ).withColumn("serious_delinquent_upb_share", col("serious_delinquent_upb") / col("active_total_upb"))

active_exposure_summary.show(truncate = False)

+-----------------+--------------------+--------------------+----------------------+--------------------+----------------------------+
|active_loan_count|active_total_upb    |delinquent_upb      |serious_delinquent_upb|delinquent_upb_share|serious_delinquent_upb_share|
+-----------------+--------------------+--------------------+----------------------+--------------------+----------------------------+
|953299           |3.086090042887791E11|3.8298080590100007E9|1.21595491683E9       |0.01240990381287216 |0.003940114837648019        |
+-----------------+--------------------+--------------------+----------------------+--------------------+----------------------------+



In [0]:
# Running reconciliation checks on the latest snapshot and UPB share calculations

latest_loan_count = latest_loan_df.count()
active_latest_loan_count = active_latest_loan_df.count()
terminated_latest_loan_count = terminated_latest_loan_df.count()

active_exposure_row = active_exposure_summary.collect()[0]

portfolio_reconciliation_df = pd.DataFrame([{
                                                "check_name" : "latest_snapshot_equals_active_plus_terminated",
                                                "expected" : float(latest_loan_count),
                                                "actual" : float(active_latest_loan_count + terminated_latest_loan_count)
                                            },
                                            {
                                                "check_name" : "latest_snapshot_equals_distinct_latest_loans","expected" : float(latest_loan_count),
                                                "actual" : float(latest_loan_df.select("loan_sequence_number").distinct().count())
                                            },
                                            {
                                                "check_name" : "delinquent_upb_share_formula",
                                                "expected" : float(active_exposure_row["delinquent_upb_share"]),
                                                "actual" : float(active_exposure_row["delinquent_upb"]) / float(active_exposure_row["active_total_upb"])
                                            },
                                            {
                                                "check_name" : "serious_delinquent_upb_share_formula",
                                                "expected" : float(active_exposure_row["serious_delinquent_upb_share"]),
                                                "actual" : float(active_exposure_row["serious_delinquent_upb"]) / float(active_exposure_row["active_total_upb"])
                                            }
])

portfolio_reconciliation_df["difference"] = (portfolio_reconciliation_df["actual"] - portfolio_reconciliation_df["expected"]
)
portfolio_reconciliation_df["passed"] = portfolio_reconciliation_df["difference"].abs() < 1e-12

portfolio_reconciliation_df

,check_name,expected,actual,difference,passed
0,latest_snapshot_equals_active_plus_terminated,1.048407e+06,1.048407e+06,0.0,True
1,latest_snapshot_equals_distinct_latest_loans,1.048407e+06,1.048407e+06,0.0,True
2,delinquent_upb_share_formula,1.240990e-02,1.240990e-02,0.0,True
3,serious_delinquent_upb_share_formula,3.940115e-03,3.940115e-03,0.0,True


In [0]:
# Comparing active delinquency risk across states
# - Showing where current loan performance is weakest at the geographic level
state_risk_summary = (active_latest_loan_df.groupBy("property_state")
                                            .agg(count("*").alias("active_loan_count"),
                                                 avg("is_delinquent").alias("active_delinquency_rate"),
                                                 avg("is_serious_delinquent").alias("active_serious_delinquency_rate")
                                            )
                                            .filter(col("active_loan_count") >= 10000)
                                            .orderBy(col("active_delinquency_rate").desc())
)

state_risk_summary.show(20, truncate = False)

+--------------+-----------------+-----------------------+-------------------------------+
|property_state|active_loan_count|active_delinquency_rate|active_serious_delinquency_rate|
+--------------+-----------------+-----------------------+-------------------------------+
|IN            |26922            |0.017123542084540526   |0.004791620236238021           |
|FL            |74394            |0.01652014947441998    |0.005806919912896201           |
|IL            |45172            |0.01613831577083149    |0.0048924112281944565          |
|MI            |36164            |0.016093352505253844   |0.0048114146665191905          |
|NY            |33478            |0.015532588565625187   |0.0041221100424159145          |
|MN            |21363            |0.015400458737068764   |0.004915040022468754           |
|CT            |10952            |0.01524835646457268    |0.004930606281957633           |
|TX            |85561            |0.014632835053353747   |0.004265962295905845           |

In [0]:
# Measuring delinquent exposure by state in dollar terms
# Showing which states contribute the largest share of risky unpaid balance

# Translating delinquent balances into state-level exposure shares
state_exposure_summary = (active_latest_loan_df.groupBy("property_state")
                                                .agg(count("*").alias("active_loan_count"),
                                                     spark_sum("current_actual_upb").alias("active_total_upb"),
                                                     spark_sum(when(col("is_delinquent") == 1, 
                                                                    col("current_actual_upb"))
                                                               .otherwise(0)).alias("delinquent_upb"),
                                                     spark_sum(when(col("is_serious_delinquent") == 1, 
                                                                    col("current_actual_upb"))
                                                               .otherwise(0)).alias("serious_delinquent_upb")
                                                )
                                                .withColumn("delinquent_upb_share", col("delinquent_upb") / col("active_total_upb")
                                                  ).withColumn("serious_delinquent_upb_share", col("serious_delinquent_upb") / col("active_total_upb"))
                                                .filter(col("active_loan_count") >= 10000)
                                                .orderBy(col("delinquent_upb_share").desc())
)

state_exposure_summary.show(20, truncate = False)

+--------------+-----------------+---------------------+--------------------+----------------------+--------------------+----------------------------+
|property_state|active_loan_count|active_total_upb     |delinquent_upb      |serious_delinquent_upb|delinquent_upb_share|serious_delinquent_upb_share|
+--------------+-----------------+---------------------+--------------------+----------------------+--------------------+----------------------------+
|FL            |74394            |2.486234734139997E10 |4.421581914599999E8 |1.6485225632000002E8  |0.01778424962810058 |0.006630599036217848        |
|NY            |33478            |1.2146699665940008E10|2.0368022975000003E8|5.645429401E7         |0.016768359748049935|0.004647706419242491        |
|CT            |10952            |3.470896919629999E9  |5.025459172E7       |1.807658481E7         |0.01447884880584618 |0.0052080442688361315       |
|IN            |26922            |5.97118464756E9      |8.346294677E7       |2.274100859000000

In [0]:
# Summarizing why terminated loans have exited the portfolio
# - Identifying whether prepayment is driving most loan terminations

termination_reason_summary = (terminated_latest_loan_df.groupBy("zero_balance_code_clean")
                                                        .agg(count("*").alias("loan_count"),
                                                             spark_sum("original_upb").alias("original_upb_sum"), 
                                                             spark_sum("current_actual_upb").alias("current_actual_upb_sum")
                                                        )
                                                        .orderBy(col("loan_count").desc())
)

termination_reason_summary.show(truncate = False)

+-----------------------+----------+----------------+----------------------+
|zero_balance_code_clean|loan_count|original_upb_sum|current_actual_upb_sum|
+-----------------------+----------+----------------+----------------------+
|01                     |91835     |3.4200645E10    |0.0                   |
|96                     |3231      |1.030488E9      |0.0                   |
|02                     |22        |4788000.0       |0.0                   |
|09                     |12        |2423000.0       |0.0                   |
|03                     |7         |2730000.0       |0.0                   |
|16                     |1         |252000.0        |0.0                   |
+-----------------------+----------+----------------+----------------------+



In [0]:
# Measuring active delinquency risk by credit score band
credit_score_risk_summary = (active_latest_loan_df.groupBy("credit_score_band")
                                                    .agg(count("*").alias("active_loan_count"),
                                                         avg("is_delinquent").alias("active_delinquency_rate"),
                                                         avg("is_serious_delinquent").alias("active_serious_delinquency_rate")
                                                    )
                                                    .orderBy("credit_score_band")
)

credit_score_risk_summary.show(truncate = False)

+-----------------+-----------------+-----------------------+-------------------------------+
|credit_score_band|active_loan_count|active_delinquency_rate|active_serious_delinquency_rate|
+-----------------+-----------------+-----------------------+-------------------------------+
|620-679          |75836            |0.06016931272746453    |0.018381771190463633           |
|680-739          |245810           |0.022187868679061064   |0.006728774256539604           |
|740-799          |508964           |0.005037684394181121   |0.0014951941591153795          |
|800+             |120668           |0.0020800875128451617  |6.0496569098684E-4             |
|<620             |1701             |0.0805408583186361     |0.021164021164021163           |
|Missing          |320              |0.021875               |0.009375                       |
+-----------------+-----------------+-----------------------+-------------------------------+



In [0]:
# Measuring delinquent and serious-delinquent UPB exposure by credit score band

# Converting delinquent balances into portfolio-style exposure shares
credit_score_exposure_summary = (active_latest_loan_df.groupBy("credit_score_band")
                                                        .agg(count("*").alias("active_loan_count"),
                                                             spark_sum("current_actual_upb").alias("active_total_upb"),spark_sum(when(col("is_delinquent") == 1, col("current_actual_upb")).otherwise(0)).alias("delinquent_upb"),
                                                             spark_sum(when(col("is_serious_delinquent") == 1, col("current_actual_upb")).otherwise(0)).alias("serious_delinquent_upb")
                                                        )
                                                        .withColumn("delinquent_upb_share", col("delinquent_upb") / col("active_total_upb")
                                                            ).withColumn("serious_delinquent_upb_share", col("serious_delinquent_upb") / col("active_total_upb"))
                                                        .orderBy("credit_score_band")
)

credit_score_exposure_summary.show(truncate = False)

+-----------------+-----------------+---------------------+--------------------+----------------------+---------------------+----------------------------+
|credit_score_band|active_loan_count|active_total_upb     |delinquent_upb      |serious_delinquent_upb|delinquent_upb_share |serious_delinquent_upb_share|
+-----------------+-----------------+---------------------+--------------------+----------------------+---------------------+----------------------------+
|620-679          |75836            |1.9831334431989975E10|1.2103708773800004E9|3.8558188908E8        |0.061033254294151185 |0.01944306321908509         |
|680-739          |245810           |7.518498158080011E10 |1.672975732529999E9 |5.279310263899999E8   |0.022251461626443023 |0.007021761730733961        |
|740-799          |508964           |1.751737722673095E11 |8.3923466469E8      |2.6907389769E8        |0.004790869396871554 |0.0015360398660559866       |
|800+             |120668           |3.7963278042289894E10|7.53163278E

In [0]:
# Measuring active delinquency risk by LTV band
ltv_risk_summary = (active_latest_loan_df.groupBy("ltv_band")
                                            .agg(count("*").alias("active_loan_count"),
                                                 avg("is_delinquent").alias("active_delinquency_rate"),
                                                 avg("is_serious_delinquent").alias("active_serious_delinquency_rate")
                                            )
                                            .filter(col("active_loan_count") >= 1000)
                                            .orderBy("ltv_band")
)

ltv_risk_summary.show(truncate = False)

+--------+-----------------+-----------------------+-------------------------------+
|ltv_band|active_loan_count|active_delinquency_rate|active_serious_delinquency_rate|
+--------+-----------------+-----------------------+-------------------------------+
|60-79   |235918           |0.01073678142405412    |0.0028738799074254614          |
|80-89   |233553           |0.010374518845829427   |0.0029886150038749234          |
|90-100  |300639           |0.02090214509760876    |0.007207980335219316           |
|<60     |183162           |0.009477948482763891   |0.0020637468470534283          |
+--------+-----------------+-----------------------+-------------------------------+



In [0]:
# Measuring delinquent and serious-delinquent UPB exposure by LTV band

# Converting delinquent balances into portfolio-style exposure shares
ltv_exposure_summary = (active_latest_loan_df.groupBy("ltv_band")
                                                .agg(count("*").alias("active_loan_count"),
                                                     spark_sum("current_actual_upb").alias("active_total_upb"),
                                                     spark_sum(when(col("is_delinquent") == 1, col("current_actual_upb"))
                                                               .otherwise(0)).alias("delinquent_upb"),
                                                     spark_sum(when(col("is_serious_delinquent") == 1, col("current_actual_upb"))
                                                               .otherwise(0)).alias("serious_delinquent_upb")
                                                )
                                                .withColumn("delinquent_upb_share", col("delinquent_upb") / col("active_total_upb")
                                                  ).withColumn("serious_delinquent_upb_share", col("serious_delinquent_upb") / col("active_total_upb"))
                                                .filter(col("active_loan_count") >= 1000)
                                                .orderBy("ltv_band")
)

ltv_exposure_summary.show(truncate = False)

+--------+-----------------+---------------------+--------------------+----------------------+--------------------+----------------------------+
|ltv_band|active_loan_count|active_total_upb     |delinquent_upb      |serious_delinquent_upb|delinquent_upb_share|serious_delinquent_upb_share|
+--------+-----------------+---------------------+--------------------+----------------------+--------------------+----------------------------+
|60-79   |235918           |7.810435928397014E10 |7.174294407299999E8 |1.9750662166000003E8  |0.009185523667399735|0.002528752856699198        |
|80-89   |233553           |8.39723035407502E10  |7.9504612242E8      |2.3688821289999998E8  |0.00946795656301341 |0.0028210279212483734       |
|90-100  |300639           |1.036029433652701E11 |1.9775041195099993E9|7.0873053472E8        |0.01908733531380442 |0.00684083397342533         |
|<60     |183162           |4.2921243627860016E10|3.3982837635E8      |7.282954755E7         |0.007917486718148556|0.0016968182045

In [0]:
# Measuring active delinquency risk by DTI band
dti_risk_summary = (active_latest_loan_df.groupBy("dti_band")
                                            .agg(count("*").alias("active_loan_count"),
                                                 avg("is_delinquent").alias("active_delinquency_rate"),
                                                 avg("is_serious_delinquent").alias("active_serious_delinquency_rate")
                                            )
                                            .orderBy("dti_band")
)

dti_risk_summary.show(truncate = False)

+--------+-----------------+-----------------------+-------------------------------+
|dti_band|active_loan_count|active_delinquency_rate|active_serious_delinquency_rate|
+--------+-----------------+-----------------------+-------------------------------+
|20-29   |147521           |0.006717687651249652   |0.0017895757214227126          |
|30-39   |284982           |0.010723484290235875   |0.0030668603631106525          |
|40-49   |425394           |0.01760250497186138    |0.005394998519020014           |
|50+     |55005            |0.022616125806744842   |0.00799927279338242            |
|<20     |40387            |0.004877807215193008   |0.0011885012504023573          |
|Missing |10               |0.0                    |0.0                            |
+--------+-----------------+-----------------------+-------------------------------+



In [0]:
# Measuring delinquent and serious-delinquent UPB exposure by DTI band

# Converting delinquent balances into portfolio-style exposure shares
dti_exposure_summary = (active_latest_loan_df.groupBy("dti_band")
                                                .agg(count("*").alias("active_loan_count"),
                                                     spark_sum("current_actual_upb").alias("active_total_upb"),
                                                     spark_sum(when(col("is_delinquent") == 1, col("current_actual_upb")).otherwise(0)).alias("delinquent_upb"),
                                                     spark_sum(when(col("is_serious_delinquent") == 1, col("current_actual_upb")).otherwise(0)).alias("serious_delinquent_upb")
                                                )
                                                .withColumn("delinquent_upb_share", col("delinquent_upb") / col("active_total_upb")
                                                  ).withColumn("serious_delinquent_upb_share", col("serious_delinquent_upb") / col("active_total_upb"))
                                                .orderBy("dti_band")
)

dti_exposure_summary.show(truncate = False)

+--------+-----------------+---------------------+--------------------+----------------------+--------------------+----------------------------+
|dti_band|active_loan_count|active_total_upb     |delinquent_upb      |serious_delinquent_upb|delinquent_upb_share|serious_delinquent_upb_share|
+--------+-----------------+---------------------+--------------------+----------------------+--------------------+----------------------------+
|20-29   |147521           |4.3194863316309944E10|2.1593233422E8      |5.879951307000001E7   |0.004999028070508239|0.0013612617000178793       |
|30-39   |284982           |9.037914032310014E10 |7.835325849200001E8 |2.3018303577000004E8  |0.008669396302276354|0.0025468602041036144       |
|40-49   |425394           |1.4452128917817032E11|2.3508999914399996E9|7.550154972799999E8   |0.0162668075050295  |0.005224251053761314        |
|50+     |55005            |2.0096266933639973E10|4.3314686947E8      |1.6039790873000002E8  |0.021553598531523162|0.0079814778167

In [0]:
# Measuring joint risk across credit score and LTV bands
credit_ltv_risk_summary = (active_latest_loan_df.groupBy("credit_score_band", "ltv_band")
                                                .agg(count("*").alias("active_loan_count"),
                                                     avg("is_delinquent").alias("active_delinquency_rate"),
                                                     avg("is_serious_delinquent").alias("active_serious_delinquency_rate"),
                                                     spark_sum("current_actual_upb").alias("active_total_upb")
                                                )
                                                .filter(col("active_loan_count") >= 1000)
                                                .orderBy("credit_score_band", "ltv_band")
)

credit_ltv_risk_summary.show(50, truncate = False)

+-----------------+--------+-----------------+-----------------------+-------------------------------+---------------------+
|credit_score_band|ltv_band|active_loan_count|active_delinquency_rate|active_serious_delinquency_rate|active_total_upb     |
+-----------------+--------+-----------------+-----------------------+-------------------------------+---------------------+
|620-679          |60-79   |19991            |0.05217347806512931    |0.01470661797809014            |5.374255921370003E9  |
|620-679          |80-89   |15133            |0.06185158263397872    |0.01810612568558779            |4.594837434450002E9  |
|620-679          |90-100  |22676            |0.07937907920268125    |0.028620568001411183           |6.388855500080005E9  |
|620-679          |<60     |18034            |0.04347343905955418    |0.009814794277475879           |3.4719214984500017E9 |
|680-739          |60-79   |54076            |0.016883645240032545   |0.00432724313928545            |1.660450639559E10    |


In [0]:
# Reshaping the credit-LTV risk summary into a matrix for easier comparison across bands
credit_ltv_pivot = (credit_ltv_risk_summary.groupBy("credit_score_band")
                                            .pivot("ltv_band", ["<60", "60-79", "80-89", "90-100"])
                                            .agg(avg("active_delinquency_rate"))
                                            .orderBy("credit_score_band")
)

credit_ltv_pivot.show(truncate = False)

+-----------------+---------------------+---------------------+---------------------+--------------------+
|credit_score_band|<60                  |60-79                |80-89                |90-100              |
+-----------------+---------------------+---------------------+---------------------+--------------------+
|620-679          |0.04347343905955418  |0.05217347806512931  |0.06185158263397872  |0.07937907920268125 |
|680-739          |0.013507647524449098 |0.016883645240032545 |0.016819081738474594 |0.03134922229813914 |
|740-799          |0.003365715597032051 |0.0036504909011318867|0.0038711214341016023|0.008177389421589855|
|800+             |0.0017831139112603643|0.001742369622686854 |0.0015347439916405434|0.00372181589994374 |
+-----------------+---------------------+---------------------+---------------------+--------------------+



In [0]:
# Measuring active delinquency risk by loan purpose
loan_purpose_risk_summary = (active_latest_loan_df.groupBy("loan_purpose")
                                                    .agg(count("*").alias("active_loan_count"),
                                                         avg("is_delinquent").alias("active_delinquency_rate"),
                                                         avg("is_serious_delinquent").alias("active_serious_delinquency_rate")
                                                    )
                                                    .orderBy(col("active_loan_count").desc())
)

loan_purpose_risk_summary.show(truncate = False)

+------------+-----------------+-----------------------+-------------------------------+
|loan_purpose|active_loan_count|active_delinquency_rate|active_serious_delinquency_rate|
+------------+-----------------+-----------------------+-------------------------------+
|P           |756694           |0.013014507845972084   |0.00405447908930162            |
|C           |113779           |0.01945877534518672    |0.005361270533226694           |
|N           |82826            |0.011035182189167653   |0.0029338613478859295          |
+------------+-----------------+-----------------------+-------------------------------+



In [0]:
# Measuring active delinquency risk by occupancy status
occupancy_risk_summary = (active_latest_loan_df.groupBy("occupancy_status")
                                                .agg(count("*").alias("active_loan_count"),
                                                     avg("is_delinquent").alias("active_delinquency_rate"),
                                                     avg("is_serious_delinquent").alias("active_serious_delinquency_rate")
                                                )
                                                .orderBy(col("active_loan_count").desc())
)

occupancy_risk_summary.show(truncate = False)

+----------------+-----------------+-----------------------+-------------------------------+
|occupancy_status|active_loan_count|active_delinquency_rate|active_serious_delinquency_rate|
+----------------+-----------------+-----------------------+-------------------------------+
|P               |873673           |0.014188374826737234   |0.0043563209576122875          |
|I               |62216            |0.007682911148257683   |0.0014947923363764948          |
|S               |17410            |0.005858701895462378   |0.001263641585295807           |
+----------------+-----------------+-----------------------+-------------------------------+



In [0]:
# Measuring early delinquency, serious delinquency and prepayment by origination quarter at month 6

# Grouping by vintage so early performance can be compared fairly across origination cohorts
month_6_vintage_summary = (month_6_status_df.groupBy("origination_quarter")
                                            .agg(count("*").alias("loan_count_observed_by_month_6"),
                                                 avg("is_delinquent").alias("month_6_delinquency_rate"),
                                                 avg("is_serious_delinquent").alias("month_6_serious_delinquency_rate"),
                                                 avg("is_prepay").alias("prepay_by_month_6_share"),
                                                 avg("is_zero_balance").alias("terminated_by_month_6_share")
                                            )
                                            .orderBy("origination_quarter")
)

month_6_vintage_summary.show(truncate = False)

+-------------------+------------------------------+------------------------+--------------------------------+-----------------------+---------------------------+
|origination_quarter|loan_count_observed_by_month_6|month_6_delinquency_rate|month_6_serious_delinquency_rate|prepay_by_month_6_share|terminated_by_month_6_share|
+-------------------+------------------------------+------------------------+--------------------------------+-----------------------+---------------------------+
|2024Q1             |214965                        |0.008080385178982625    |0.001279278022003582            |0.03366594561905426    |0.03550345405065941        |
|2024Q2             |255487                        |0.009703037727947018    |0.0016987165687490948           |0.040158599067662934   |0.04167726733649853        |
|2024Q3             |295847                        |0.007074602750746162    |0.0012371259468576662           |0.026293996559032203   |0.02726071246286087        |
|2024Q4             |2

In [0]:
# Comparing which loan purposes and credit bands are prepaying fastest within the first 6 months
prepay_by_month_6_loan_purpose_summary = (month_6_status_df.groupBy("loan_purpose")
                                                            .agg(count("*").alias("loan_count"),
                                                                 avg("is_prepay").alias("prepay_by_month_6_share")
                                                            )
                                                            .orderBy(col("prepay_by_month_6_share").desc())
)

prepay_by_month_6_credit_score_summary = (month_6_status_df.groupBy("credit_score_band")
                                                            .agg(count("*").alias("loan_count"),
                                                                 avg("is_prepay").alias("prepay_by_month_6_share")
                                                            )
                                                            .orderBy(col("prepay_by_month_6_share").desc())
)

prepay_by_month_6_loan_purpose_summary.show(truncate = False)
prepay_by_month_6_credit_score_summary.show(truncate = False)

+------------+----------+-----------------------+
|loan_purpose|loan_count|prepay_by_month_6_share|
+------------+----------+-----------------------+
|C           |130386    |0.03528753086987867    |
|P           |827197    |0.030973274806364144   |
|N           |90354     |0.025344755074484804   |
+------------+----------+-----------------------+

+-----------------+----------+-----------------------+
|credit_score_band|loan_count|prepay_by_month_6_share|
+-----------------+----------+-----------------------+
|800+             |138291    |0.06084271572264283    |
|740-799          |558666    |0.031730944786330297   |
|Missing          |360       |0.030555555555555555   |
|<620             |1883      |0.01858736059479554    |
|680-739          |265699    |0.018306429455888052   |
|620-679          |83038     |0.017594354391965123   |
+-----------------+----------+-----------------------+



In [0]:
# Tracking how active portfolio delinquency and delinquent UPB are changing over monthly reporting periods
monthly_active_trend_summary = (analysis_df.filter(col("is_zero_balance") == 0)
                                            .groupBy("monthly_reporting_period")
                                            .agg(count("*").alias("active_record_count"),
                                                 avg("is_delinquent").alias("active_delinquency_rate"),
                                                 avg("is_serious_delinquent").alias("active_serious_delinquency_rate"),
                                                 spark_sum("current_actual_upb").alias("active_total_upb"),
                                                 spark_sum(when(col("is_delinquent") == 1, col("current_actual_upb"))
                                                           .otherwise(0)).alias("delinquent_upb")
                                            )
                                            .withColumn("delinquent_upb_share", col("delinquent_upb") / col("active_total_upb"))
                                            .orderBy("monthly_reporting_period")
)

monthly_active_trend_summary.show(30, truncate = False)

+------------------------+-------------------+-----------------------+-------------------------------+---------------------+--------------------+---------------------+
|monthly_reporting_period|active_record_count|active_delinquency_rate|active_serious_delinquency_rate|active_total_upb     |delinquent_upb      |delinquent_upb_share |
+------------------------+-------------------+-----------------------+-------------------------------+---------------------+--------------------+---------------------+
|2024-01-01              |498                |0.0                    |0.0                            |1.54151E8            |0.0                 |0.0                  |
|2024-02-01              |54373              |1.1034888639582146E-4  |0.0                            |1.7312468E10         |1663000.0           |9.605793928398886E-5 |
|2024-03-01              |126276             |0.0024549399727580856  |0.0                            |4.124886444539E10    |8.8145E7            |0.0021369073109

In [0]:
# Summarizing how borrower and loan quality are differing across origination quarters
origination_profile_summary = (latest_loan_df.groupBy("origination_quarter")
                                                .agg(count("*").alias("loan_count"),
                                                     avg("credit_score_clean").alias("avg_credit_score"),
                                                     avg("loan_to_value_clean").alias("avg_ltv"),
                                                     avg("debt_to_income_ratio_clean").alias("avg_dti"),
                                                     avg(when(col("credit_score_clean") < 680, 1)
                                                         .otherwise(0)).alias("share_credit_below_680"),
                                                     avg(when(col("loan_to_value_clean") >= 90, 1)
                                                         .otherwise(0)).alias("share_ltv_90_plus"),
                                                     avg(when(col("debt_to_income_ratio_clean") >= 45, 1)
                                                         .otherwise(0)).alias("share_dti_45_plus"),
                                                     avg(when(col("first_time_homebuyer_flag") == "Y", 1)
                                                         .otherwise(0)).alias("share_first_time_homebuyer")
                                                )
                                                .orderBy("origination_quarter")
)

origination_profile_summary.show(truncate = False)

+-------------------+----------+-----------------+-----------------+------------------+----------------------+-------------------+-------------------+--------------------------+
|origination_quarter|loan_count|avg_credit_score |avg_ltv          |avg_dti           |share_credit_below_680|share_ltv_90_plus  |share_dti_45_plus  |share_first_time_homebuyer|
+-------------------+----------+-----------------+-----------------+------------------+----------------------+-------------------+-------------------+--------------------------+
|2024Q1             |215022    |751.567733652142 |75.18763661392788|37.95932452480943 |0.0808847466770842    |0.32065556082633406|0.3022109365553292 |0.38917878170605796       |
|2024Q2             |255625    |751.5188262412875|74.99712467188007|38.05126770701708 |0.08703374083129585   |0.31918630806845966|0.3070044009779951 |0.40472176039119806       |
|2024Q3             |296015    |752.493590935051 |74.86220491061174|37.716559745686844|0.08389777545056838   |

In [0]:
# Ranking combined credit, LTV and DTI segments to identify the riskiest borrower pockets

# Combining borrower quality, leverage and payment burden to capture interaction-based risk
segment_risk_summary = (active_latest_loan_df.groupBy("credit_score_band", "ltv_band", "dti_band")
                                                .agg(count("*").alias("active_loan_count"),
                                                     avg("is_delinquent").alias("active_delinquency_rate"),
                                                     avg("is_serious_delinquent").alias("active_serious_delinquency_rate"),
                                                     spark_sum("current_actual_upb").alias("active_total_upb")
                                                )
                                                .filter(col("active_loan_count") >= 1000)
                                                .orderBy(col("active_delinquency_rate").desc(), col("active_loan_count").desc())
)

segment_risk_summary.show(30, truncate = False)

+-----------------+--------+--------+-----------------+-----------------------+-------------------------------+--------------------+
|credit_score_band|ltv_band|dti_band|active_loan_count|active_delinquency_rate|active_serious_delinquency_rate|active_total_upb    |
+-----------------+--------+--------+-----------------+-----------------------+-------------------------------+--------------------+
|620-679          |90-100  |50+     |1328             |0.09864457831325302    |0.04292168674698795            |4.486985779000001E8 |
|620-679          |90-100  |40-49   |12792            |0.08137898686679175    |0.02931519699812383            |3.9161297727199993E9|
|620-679          |80-89   |50+     |1183             |0.07776838546069316    |0.01944209636517329            |4.2487090602E8      |
|620-679          |90-100  |30-39   |6479             |0.07593764469825591    |0.02592992745794104            |1.6028633190500004E9|
|620-679          |90-100  |20-29   |1820             |0.065384615384

In [0]:
# Combining state-level rate and exposure views to highlight where delinquent UPB is concentrating geographically
top_state_risk_insight = (state_exposure_summary.join(state_risk_summary
                                                      .select("property_state", "active_loan_count",
                                                              "active_delinquency_rate",
                                                              "active_serious_delinquency_rate"
                                                    ),
                                                    on = ["property_state", "active_loan_count"],
                                                    how = "inner"
                                                )
                                                .select("property_state", "active_loan_count",
                                                        "active_total_upb", "active_delinquency_rate",
                                                        "active_serious_delinquency_rate",
                                                        "delinquent_upb_share",
                                                        "serious_delinquent_upb_share"
                                                )
                                                .orderBy(col("delinquent_upb_share").desc(), 
                                                         col("active_total_upb").desc()
                                                )
)

top_state_risk_insight.show(20, truncate = False)

+--------------+-----------------+---------------------+-----------------------+-------------------------------+--------------------+----------------------------+
|property_state|active_loan_count|active_total_upb     |active_delinquency_rate|active_serious_delinquency_rate|delinquent_upb_share|serious_delinquent_upb_share|
+--------------+-----------------+---------------------+-----------------------+-------------------------------+--------------------+----------------------------+
|FL            |74394            |2.486234734139997E10 |0.01652014947441998    |0.005806919912896201           |0.01778424962810058 |0.006630599036217848        |
|NY            |33478            |1.2146699665940008E10|0.015532588565625187   |0.0041221100424159145          |0.016768359748049935|0.004647706419242491        |
|CT            |10952            |3.470896919629999E9  |0.01524835646457268    |0.004930606281957633           |0.01447884880584618 |0.0052080442688361315       |
|IN            |26922 

In [0]:
# Pulling the highest-risk borrower segments into a focused view for business interpretation
top_segment_risk_insight = (segment_risk_summary.select("credit_score_band", "ltv_band", "dti_band",
                                                        "active_loan_count", "active_total_upb","active_delinquency_rate", "active_serious_delinquency_rate"
                                                )
                                                .orderBy(col("active_delinquency_rate").desc(), col("active_loan_count").desc())
)

top_segment_risk_insight.show(30, truncate = False)

+-----------------+--------+--------+-----------------+--------------------+-----------------------+-------------------------------+
|credit_score_band|ltv_band|dti_band|active_loan_count|active_total_upb    |active_delinquency_rate|active_serious_delinquency_rate|
+-----------------+--------+--------+-----------------+--------------------+-----------------------+-------------------------------+
|620-679          |90-100  |50+     |1328             |4.486985779000001E8 |0.09864457831325302    |0.04292168674698795            |
|620-679          |90-100  |40-49   |12792            |3.9161297727199993E9|0.08137898686679175    |0.02931519699812383            |
|620-679          |80-89   |50+     |1183             |4.2487090602E8      |0.07776838546069316    |0.01944209636517329            |
|620-679          |90-100  |30-39   |6479             |1.6028633190500004E9|0.07593764469825591    |0.02592992745794104            |
|620-679          |90-100  |20-29   |1820             |3.681562177900

In [0]:
# Identifying the largest segments by exposure so risk can be viewed alongside portfolio size

largest_segment_exposure = (segment_risk_summary.select("credit_score_band", "ltv_band", "dti_band",
                                                        "active_loan_count", "active_total_upb",
                                                        "active_delinquency_rate", "active_serious_delinquency_rate"
                                                )
                                                .orderBy(col("active_total_upb").desc())
)

largest_segment_exposure.show(30, truncate = False)

+-----------------+--------+--------+-----------------+---------------------+-----------------------+-------------------------------+
|credit_score_band|ltv_band|dti_band|active_loan_count|active_total_upb     |active_delinquency_rate|active_serious_delinquency_rate|
+-----------------+--------+--------+-----------------+---------------------+-----------------------+-------------------------------+
|740-799          |90-100  |40-49   |75970            |2.872942721772E10    |0.010082927471370278   |0.003422403580360669           |
|740-799          |80-89   |40-49   |55643            |2.1831640662699974E10|0.00512193806947864    |0.001365850151860971           |
|680-739          |90-100  |40-49   |55607            |1.871295771546E10    |0.03546316111280954    |0.011545309043825417           |
|740-799          |60-79   |40-49   |51239            |1.8488802077439987E10|0.005152325377154121   |0.0013271141122972736          |
|740-799          |90-100  |30-39   |49188            |1.75939

In [0]:
# Building a credit risk ladder to show how delinquent exposure is changing across credit score bands

credit_story_table = (credit_score_exposure_summary.withColumn("band_order",
                                                               when(col("credit_score_band") == "<620", 1)
                                                               .when(col("credit_score_band") == "620-679", 2)
                                                               .when(col("credit_score_band") == "680-739", 3)
                                                               .when(col("credit_score_band") == "740-799", 4)
                                                               .when(col("credit_score_band") == "800+", 5)
                                                               .otherwise(6)
                                                    )
                                                    .select("credit_score_band", "active_loan_count",
                                                            "active_total_upb", "delinquent_upb_share",
                                                            "serious_delinquent_upb_share", "band_order"
                                                    )
                                                    .orderBy("band_order")
                                                    .drop("band_order")
)

credit_story_table.show(truncate = False)

+-----------------+-----------------+---------------------+---------------------+----------------------------+
|credit_score_band|active_loan_count|active_total_upb     |delinquent_upb_share |serious_delinquent_upb_share|
+-----------------+-----------------+---------------------+---------------------+----------------------------+
|<620             |1701             |3.857100426700001E8  |0.07982097338424997  |0.02304394598717903         |
|620-679          |75836            |1.9831334431989975E10|0.061033254294151185 |0.01944306321908509         |
|680-739          |245810           |7.518498158080011E10 |0.022251461626443023 |0.007021761730733961        |
|740-799          |508964           |1.751737722673095E11 |0.004790869396871554 |0.0015360398660559866       |
|800+             |120668           |3.7963278042289894E10|0.0019839258273771825|6.349903502312505E-4        |
|Missing          |320              |6.992792372E7        |0.016055182254451756 |0.005341314887248307        |
+

In [0]:
# Building an LTV risk ladder to show how delinquent exposure is changing as leverage increases
ltv_story_table = (ltv_exposure_summary.withColumn("band_order",
                                                   when(col("ltv_band") == "<60", 1)
                                                   .when(col("ltv_band") == "60-79", 2)
                                                   .when(col("ltv_band") == "80-89", 3)
                                                   .when(col("ltv_band") == "90-100", 4)
                                                   .when(col("ltv_band") == "100+", 5)
                                                   .otherwise(6)
                                        )
                                        .select("ltv_band", "active_loan_count", "active_total_upb",            
                                                "delinquent_upb_share", "serious_delinquent_upb_share", "band_order"
                                        )
                                        .orderBy("band_order")
                                        .drop("band_order")
)

ltv_story_table.show(truncate = False)

+--------+-----------------+---------------------+--------------------+----------------------------+
|ltv_band|active_loan_count|active_total_upb     |delinquent_upb_share|serious_delinquent_upb_share|
+--------+-----------------+---------------------+--------------------+----------------------------+
|<60     |183162           |4.2921243627860016E10|0.007917486718148556|0.001696818204557489        |
|60-79   |235918           |7.810435928397014E10 |0.009185523667399735|0.002528752856699198        |
|80-89   |233553           |8.39723035407502E10  |0.00946795656301341 |0.0028210279212483734       |
|90-100  |300639           |1.036029433652701E11 |0.01908733531380442 |0.00684083397342533         |
+--------+-----------------+---------------------+--------------------+----------------------------+



In [0]:
# Measuring how delinquent exposure is changing across debt-to-income bands

dti_story_table = (dti_exposure_summary.withColumn("band_order",
                                                   when(col("dti_band") == "<20", 1)
                                                   .when(col("dti_band") == "20-29", 2)
                                                   .when(col("dti_band") == "30-39", 3)
                                                   .when(col("dti_band") == "40-49", 4)
                                                   .when(col("dti_band") == "50+", 5)
                                                   .otherwise(6)
                                        )
                                        .select("dti_band", "active_loan_count", "active_total_upb",
                                                "delinquent_upb_share", "serious_delinquent_upb_share", "band_order"
                                        )
                                        .orderBy("band_order")
                                        .drop("band_order")
)

dti_story_table.show(truncate = False)

+--------+-----------------+---------------------+--------------------+----------------------------+
|dti_band|active_loan_count|active_total_upb     |delinquent_upb_share|serious_delinquent_upb_share|
+--------+-----------------+---------------------+--------------------+----------------------------+
|<20     |40387            |1.0414394857030003E10|0.004445412296687478|0.0011099024128317341       |
|20-29   |147521           |4.3194863316309944E10|0.004999028070508239|0.0013612617000178793       |
|30-39   |284982           |9.037914032310014E10 |0.008669396302276354|0.0025468602041036144       |
|40-49   |425394           |1.4452128917817032E11|0.0162668075050295  |0.005224251053761314        |
|50+     |55005            |2.0096266933639973E10|0.021553598531523162|0.007981477816733381        |
|Missing |10               |3049680.5300000003   |0.0                 |0.0                         |
+--------+-----------------+---------------------+--------------------+--------------------

In [0]:
# Identifying which credit-score and loan-purpose combinations are prepaying fastest within the first 6 months
early_prepay_segment_summary = (month_6_status_df.groupBy("credit_score_band", "loan_purpose")
                                                    .agg(count("*").alias("loan_count"),
                                                         avg("is_prepay").alias("prepay_by_month_6_share"),
                                                         avg("is_zero_balance").alias("terminated_by_month_6_share")
                                                    )
                                                    .filter(col("loan_count") >= 1000)
                                                    .orderBy(col("prepay_by_month_6_share").desc(), 
                                                             col("loan_count").desc())
)

early_prepay_segment_summary.show(30, truncate = False)

+-----------------+------------+----------+-----------------------+---------------------------+
|credit_score_band|loan_purpose|loan_count|prepay_by_month_6_share|terminated_by_month_6_share|
+-----------------+------------+----------+-----------------------+---------------------------+
|800+             |P           |116639    |0.06385514279100472    |0.06468676857654815        |
|800+             |C           |10268     |0.06310868718348267    |0.06388780677834048        |
|740-799          |C           |50435     |0.039179141469217804   |0.040229999008624964       |
|740-799          |P           |456079    |0.03156251438895454    |0.032619348840880635       |
|680-739          |C           |45882     |0.028813042151606294   |0.030098949479098556       |
|800+             |N           |11384     |0.02793394237526353    |0.028636683063949404       |
|620-679          |C           |23041     |0.027776572197387266   |0.02912199991319821        |
|740-799          |N           |52152   

In [0]:
# Summarizing vintage risk 
# - each origination quarter can be compared on delinquency, prepayment and borrower profile mix
vintage_risk_profile_summary = (month_6_vintage_summary.join(origination_profile_summary, 
                                                             on = "origination_quarter",
                                                             how = "inner")
                                                        .select("origination_quarter", "loan_count_observed_by_month_6",
                                                                "month_6_delinquency_rate",
                                                                "month_6_serious_delinquency_rate",
                                                                "prepay_by_month_6_share", "avg_credit_score",
                                                                "avg_ltv", "avg_dti", "share_credit_below_680","share_ltv_90_plus", "share_dti_45_plus",
                                                                "share_first_time_homebuyer"
                                                        )
                                                        .orderBy("origination_quarter")
)

vintage_risk_profile_summary.show(truncate = False)

+-------------------+------------------------------+------------------------+--------------------------------+-----------------------+-----------------+-----------------+------------------+----------------------+-------------------+-------------------+--------------------------+
|origination_quarter|loan_count_observed_by_month_6|month_6_delinquency_rate|month_6_serious_delinquency_rate|prepay_by_month_6_share|avg_credit_score |avg_ltv          |avg_dti           |share_credit_below_680|share_ltv_90_plus  |share_dti_45_plus  |share_first_time_homebuyer|
+-------------------+------------------------------+------------------------+--------------------------------+-----------------------+-----------------+-----------------+------------------+----------------------+-------------------+-------------------+--------------------------+
|2024Q1             |214965                        |0.008080385178982625    |0.001279278022003582            |0.03366594561905426    |751.567733652142 |75.18763

In [0]:
# Converting portfolio-level delinquent UPB into state-level contribution shares 
# - which states are driving the portfolio's stressed exposure
portfolio_delinquent_upb = active_exposure_summary.collect()[0]["delinquent_upb"]
portfolio_serious_delinquent_upb = active_exposure_summary.collect()[0]["serious_delinquent_upb"]

state_contribution_summary = (state_exposure_summary.withColumn("share_of_portfolio_delinquent_upb", 
                                                                col("delinquent_upb") / portfolio_delinquent_upb
                                                    )
                                                    .withColumn("share_of_portfolio_serious_delinquent_upb",
                                                        col("serious_delinquent_upb") / portfolio_serious_delinquent_upb
                                                    )
                                                    .select("property_state", "active_loan_count",
                                                            "active_total_upb", "delinquent_upb",
                                                            "serious_delinquent_upb",
                                                            "delinquent_upb_share", "serious_delinquent_upb_share",
                                                            "share_of_portfolio_delinquent_upb",
                                                            "share_of_portfolio_serious_delinquent_upb"
                                                    )
                                                    .filter(col("active_loan_count") >= 10000)
                                                    .orderBy(col("share_of_portfolio_delinquent_upb").desc())
)

state_contribution_summary.show(20, truncate = False)

+--------------+-----------------+---------------------+--------------------+----------------------+--------------------+----------------------------+---------------------------------+-----------------------------------------+
|property_state|active_loan_count|active_total_upb     |delinquent_upb      |serious_delinquent_upb|delinquent_upb_share|serious_delinquent_upb_share|share_of_portfolio_delinquent_upb|share_of_portfolio_serious_delinquent_upb|
+--------------+-----------------+---------------------+--------------------+----------------------+--------------------+----------------------------+---------------------------------+-----------------------------------------+
|FL            |74394            |2.486234734139997E10 |4.421581914599999E8 |1.6485225632000002E8  |0.01778424962810058 |0.006630599036217848        |0.11545178887484432              |0.1355743161512687                       |
|CA            |69537            |3.4262030171869934E10|3.9653347074E8      |1.4789044333E8 

In [0]:
# Converting portfolio-level delinquent UPB into segment-level contribution shares
# - which borrower segments are driving the portfolio's stressed exposure
segment_exposure_contribution_summary = (
    active_latest_loan_df
    .groupBy("credit_score_band", "ltv_band", "dti_band")
    .agg(
        count("*").alias("active_loan_count"),
        spark_sum("current_actual_upb").alias("active_total_upb"),
        spark_sum(
            when(col("is_delinquent") == 1, col("current_actual_upb")).otherwise(0)
        ).alias("delinquent_upb"),
        spark_sum(
            when(col("is_serious_delinquent") == 1, col("current_actual_upb")).otherwise(0)
        ).alias("serious_delinquent_upb")
    )
    .withColumn(
        "share_of_portfolio_delinquent_upb",
        col("delinquent_upb") / portfolio_delinquent_upb
    )
    .withColumn(
        "share_of_portfolio_serious_delinquent_upb",
        col("serious_delinquent_upb") / portfolio_serious_delinquent_upb
    )
    .filter(col("active_loan_count") >= 1000)
    .orderBy(col("share_of_portfolio_delinquent_upb").desc(), 
             col("active_loan_count").desc())
)

segment_exposure_contribution_summary.show(30, truncate = False)

+-----------------+--------+--------+-----------------+---------------------+--------------------+----------------------+---------------------------------+-----------------------------------------+
|credit_score_band|ltv_band|dti_band|active_loan_count|active_total_upb     |delinquent_upb      |serious_delinquent_upb|share_of_portfolio_delinquent_upb|share_of_portfolio_serious_delinquent_upb|
+-----------------+--------+--------+-----------------+---------------------+--------------------+----------------------+---------------------------------+-----------------------------------------+
|680-739          |90-100  |40-49   |55607            |1.871295771546E10    |6.432444255899997E8 |2.1853864226000002E8  |0.16795735339182438              |0.17972594150918955                      |
|620-679          |90-100  |40-49   |12792            |3.9161297727199993E9 |3.2277276268E8      |1.1695008047000001E8  |0.08427909642120192              |0.09617961887508907                      |
|740-799  

In [0]:
# Pulling key portfolio exposure values into one row

active_exposure_row = active_exposure_summary.collect()[0]

portfolio_delinquent_upb = float(active_exposure_row["delinquent_upb"])
portfolio_serious_delinquent_upb = float(active_exposure_row["serious_delinquent_upb"])
portfolio_delinquent_upb_share = float(active_exposure_row["delinquent_upb_share"])
portfolio_serious_delinquent_upb_share = float(active_exposure_row["serious_delinquent_upb_share"])

portfolio_exposure_benchmarks = pd.DataFrame([
                                                {
                                                    "portfolio_delinquent_upb" : portfolio_delinquent_upb,
                                                    "portfolio_serious_delinquent_upb" : portfolio_serious_delinquent_upb,
                                                    "portfolio_delinquent_upb_share" : portfolio_delinquent_upb_share,
                                                    "portfolio_serious_delinquent_upb_share" : portfolio_serious_delinquent_upb_share
                                                }
])

portfolio_exposure_benchmarks

,portfolio_delinquent_upb,portfolio_serious_delinquent_upb,portfolio_delinquent_upb_share,portfolio_serious_delinquent_upb_share
0,3.829808e+09,1.215955e+09,0.01241,0.00394


In [0]:
# Building a segment concentration watchlist by combining segment size with segment contribution to delinquent exposure
segment_concentration_watchlist = (segment_exposure_contribution_summary.select("credit_score_band", "ltv_band",
                                                                                "dti_band", "active_loan_count",
                                                                                "active_total_upb", "delinquent_upb",
                                                                                "serious_delinquent_upb",
                                                                                "share_of_portfolio_delinquent_upb",
                                                                                "share_of_portfolio_serious_delinquent_upb"
                                                                        )
                                                                        .orderBy(col("share_of_portfolio_delinquent_upb").desc(),
                                                                                 col("active_total_upb").desc()
                                                                        )
)

segment_concentration_watchlist.show(20, truncate = False)

+-----------------+--------+--------+-----------------+---------------------+--------------------+----------------------+---------------------------------+-----------------------------------------+
|credit_score_band|ltv_band|dti_band|active_loan_count|active_total_upb     |delinquent_upb      |serious_delinquent_upb|share_of_portfolio_delinquent_upb|share_of_portfolio_serious_delinquent_upb|
+-----------------+--------+--------+-----------------+---------------------+--------------------+----------------------+---------------------------------+-----------------------------------------+
|680-739          |90-100  |40-49   |55607            |1.871295771546E10    |6.432444255899997E8 |2.1853864226000002E8  |0.16795735339182438              |0.17972594150918955                      |
|620-679          |90-100  |40-49   |12792            |3.9161297727199993E9 |3.2277276268E8      |1.1695008047000001E8  |0.08427909642120192              |0.09617961887508907                      |
|740-799  

In [0]:
# Building a state concentration watchlist by combining state size with state contribution to delinquent exposure
state_concentration_watchlist = (state_contribution_summary.select("property_state", "active_loan_count",
                                                                   "active_total_upb", "delinquent_upb",
                                                                   "serious_delinquent_upb",
                                                                   "share_of_portfolio_delinquent_upb",
                                                                   "share_of_portfolio_serious_delinquent_upb"
                                                            )
                                                            .orderBy(col("share_of_portfolio_delinquent_upb").desc(), 
                                                                     col("active_total_upb").desc()
    )
)

state_concentration_watchlist.show(20, truncate = False)

+--------------+-----------------+---------------------+--------------------+----------------------+---------------------------------+-----------------------------------------+
|property_state|active_loan_count|active_total_upb     |delinquent_upb      |serious_delinquent_upb|share_of_portfolio_delinquent_upb|share_of_portfolio_serious_delinquent_upb|
+--------------+-----------------+---------------------+--------------------+----------------------+---------------------------------+-----------------------------------------+
|FL            |74394            |2.486234734139997E10 |4.421581914599999E8 |1.6485225632000002E8  |0.11545178887484432              |0.1355743161512687                       |
|CA            |69537            |3.4262030171869934E10|3.9653347074E8      |1.4789044333E8        |0.10353873213231038              |0.1216249396117013                       |
|TX            |85561            |2.802095745280001E10 |3.8335950134000003E8|1.1510396064E8        |0.1000988810491

In [0]:
# Converting portfolio-level delinquency rates into benchmark values
# - segment and state risk can be judged against the overall portfolio

portfolio_active_delinquency_rate = active_portfolio_summary.collect()[0]["active_delinquency_rate"]
portfolio_serious_delinquency_rate = active_portfolio_summary.collect()[0]["active_serious_delinquency_rate"]

elevated_risk_state_watchlist = (state_contribution_summary.withColumn("delinquent_upb_risk_index",
                                                                       col("delinquent_upb_share") / portfolio_delinquent_upb_share
                                                            )
                                                            .withColumn("serious_delinquent_upb_risk_index",
                                                                        col("serious_delinquent_upb_share") / portfolio_serious_delinquent_upb_share
                                                            )
                                                            .filter(
                                                                ((col("share_of_portfolio_delinquent_upb") >= 0.02) &
                                                                 (col("delinquent_upb_risk_index") >= 1.10)) |
                                                                ((col("share_of_portfolio_serious_delinquent_upb") >= 0.02) &
                                                                 (col("serious_delinquent_upb_risk_index") >= 1.10))
                                                            )
                                                            .select("property_state", "active_loan_count",
                                                                    "active_total_upb", "delinquent_upb_share",
                                                                    "serious_delinquent_upb_share",
                                                                    "share_of_portfolio_delinquent_upb",
                                                                    "share_of_portfolio_serious_delinquent_upb",
                                                                    "delinquent_upb_risk_index",
                                                                    "serious_delinquent_upb_risk_index"
                                                            )
                                                            .orderBy(col("delinquent_upb_risk_index").desc(),
                                                                     col("share_of_portfolio_delinquent_upb").desc()
                                                            )
)

elevated_risk_state_watchlist.show(20, truncate = False)

+--------------+-----------------+---------------------+--------------------+----------------------------+---------------------------------+-----------------------------------------+-------------------------+---------------------------------+
|property_state|active_loan_count|active_total_upb     |delinquent_upb_share|serious_delinquent_upb_share|share_of_portfolio_delinquent_upb|share_of_portfolio_serious_delinquent_upb|delinquent_upb_risk_index|serious_delinquent_upb_risk_index|
+--------------+-----------------+---------------------+--------------------+----------------------------+---------------------------------+-----------------------------------------+-------------------------+---------------------------------+
|FL            |74394            |2.486234734139997E10 |0.01778424962810058 |0.006630599036217848        |0.11545178887484432              |0.1355743161512687                       |1.433069095157199        |1.6828441071976128               |
|NY            |33478       

In [0]:
# Filtering for the borrower segments that are both large enough to matter and riskier than the overall portfolio

elevated_risk_segment_watchlist = (segment_exposure_contribution_summary.withColumn("delinquent_upb_risk_index",
                                                                                    (col("delinquent_upb") / col("active_total_upb")) / portfolio_delinquent_upb_share
                                                                        )
                                                                        .withColumn("serious_delinquent_upb_risk_index",
                                                                                    (col("serious_delinquent_upb") / col("active_total_upb")) / portfolio_serious_delinquent_upb_share
                                                                        )
                                                                        .filter(
                                                                            (
                                                                                (col("active_loan_count") >= 3000) & 
                                                                                (col("share_of_portfolio_delinquent_upb") >= 0.01) &
                                                                                (col("delinquent_upb_risk_index") >= 1.25)
                                                                            ) | 
                                                                            (
                                                                                (col("active_loan_count") >= 3000) &
                                                                                (col("share_of_portfolio_serious_delinquent_upb") >= 0.01) &
                                                                                (col("serious_delinquent_upb_risk_index") >= 1.25)
                                                                            )
                                                                        )
                                                                        .select("credit_score_band", "ltv_band",
                                                                                "dti_band", "active_loan_count",
                                                                                "active_total_upb", "delinquent_upb",
                                                                                "serious_delinquent_upb",
                                                                                "share_of_portfolio_delinquent_upb",
                                                                                "share_of_portfolio_serious_delinquent_upb", "delinquent_upb_risk_index",
                                                                                "serious_delinquent_upb_risk_index"
                                                                        )
                                                                        .orderBy(col("delinquent_upb_risk_index").desc(), 
                                                                                 col("share_of_portfolio_delinquent_upb").desc()
                                                                        )
)

elevated_risk_segment_watchlist.show(20, truncate = False)

+-----------------+--------+--------+-----------------+--------------------+--------------------+----------------------+---------------------------------+-----------------------------------------+-------------------------+---------------------------------+
|credit_score_band|ltv_band|dti_band|active_loan_count|active_total_upb    |delinquent_upb      |serious_delinquent_upb|share_of_portfolio_delinquent_upb|share_of_portfolio_serious_delinquent_upb|delinquent_upb_risk_index|serious_delinquent_upb_risk_index|
+-----------------+--------+--------+-----------------+--------------------+--------------------+----------------------+---------------------------------+-----------------------------------------+-------------------------+---------------------------------+
|620-679          |90-100  |40-49   |12792            |3.9161297727199993E9|3.2277276268E8      |1.1695008047000001E8  |0.08427909642120192              |0.09617961887508907                      |6.641579707109664        |7.57939

In [0]:
# Creating active loan history only and grouping delinquency into business-friendly buckets for transition tracking

print("Transition scope note : the transition analysis below uses consecutive month-to-month "
      "observations for loans that remain active at both observations. Zero-balance exits are "
      "excluded before transition construction."
)

# Converting monthly delinquency values into simple transition states : Current, 30D, 60D and 90D+.
active_history_df = (analysis_df.filter(col("is_zero_balance") == 0)
                                .withColumn("delinquency_bucket",
                                            when(col("delinquency_months").isNull() | (col("delinquency_months") == 0), "Current")
                                            .when(col("delinquency_months") == 1, "30D")
                                            .when(col("delinquency_months") == 2, "60D")
                                            .otherwise("90D+")
                                )
)

active_history_df.groupBy("delinquency_bucket").count().orderBy("count", ascending = False).show(truncate = False)

Transition scope note : the transition analysis below uses consecutive month-to-month observations for loans that remain active at both observations. Zero-balance exits are excluded before transition construction.
+------------------+--------+
|delinquency_bucket|count   |
+------------------+--------+
|Current           |14008738|
|30D               |79041   |
|90D+              |26767   |
|60D               |17319   |
+------------------+--------+



In [0]:
# Creating consecutive month-to-month loan transitions 
# - how delinquency is improving or worsening

transition_window = Window.partitionBy("loan_sequence_number").orderBy("monthly_reporting_period")

loan_transitions_df = (active_history_df.withColumn("next_reporting_period", 
                                                    lead("monthly_reporting_period")
                                                    .over(transition_window))
                                        .withColumn("next_delinquency_bucket", 
                                                    lead("delinquency_bucket")
                                                    .over(transition_window))
                                        .withColumn("next_is_delinquent", 
                                                    lead("is_delinquent")
                                                    .over(transition_window))
                                        .withColumn("next_is_serious_delinquent", 
                                                    lead("is_serious_delinquent")
                                                    .over(transition_window))
                                        .filter(col("next_reporting_period").isNotNull())
                                        .filter(add_months(col("monthly_reporting_period"), 1) == col("next_reporting_period"))
)

print("Transition row count (active-to-active consecutive monthly observations only) : ", loan_transitions_df.count())
loan_transitions_df.select("loan_sequence_number", "monthly_reporting_period",
                           "delinquency_bucket", "next_reporting_period", "next_delinquency_bucket"
).show(10, truncate = False)

Transition row count (active-to-active consecutive monthly observations only) :  13085095
+--------------------+------------------------+------------------+---------------------+-----------------------+
|loan_sequence_number|monthly_reporting_period|delinquency_bucket|next_reporting_period|next_delinquency_bucket|
+--------------------+------------------------+------------------+---------------------+-----------------------+
|F24Q10000006        |2024-02-01              |Current           |2024-03-01           |Current                |
|F24Q10000006        |2024-03-01              |Current           |2024-04-01           |Current                |
|F24Q10000006        |2024-04-01              |Current           |2024-05-01           |Current                |
|F24Q10000006        |2024-05-01              |Current           |2024-06-01           |Current                |
|F24Q10000006        |2024-06-01              |Current           |2024-07-01           |Current                |
|F24Q1

In [0]:
# Summarizing how loans are moving from one delinquency bucket to the next month’s bucket

transition_matrix_summary = (loan_transitions_df.groupBy("delinquency_bucket", "next_delinquency_bucket")
                                                .agg(count("*").alias("transition_count"))
)

transition_totals = (loan_transitions_df.groupBy("delinquency_bucket")
                                        .agg(count("*").alias("from_bucket_total"))
)

transition_matrix_summary = (transition_matrix_summary.join(transition_totals, 
                                                            on = "delinquency_bucket", 
                                                            how = "inner")
                                                      .withColumn("transition_rate", col("transition_count") / col("from_bucket_total"))
                                                      .orderBy("delinquency_bucket", col("transition_rate").desc())
)

transition_matrix_summary.show(20, truncate = False)

+------------------+-----------------------+----------------+-----------------+---------------------+
|delinquency_bucket|next_delinquency_bucket|transition_count|from_bucket_total|transition_rate      |
+------------------+-----------------------+----------------+-----------------+---------------------+
|30D               |Current                |35539           |71035            |0.5003026676990217   |
|30D               |30D                    |22646           |71035            |0.31880059125783067  |
|30D               |60D                    |12684           |71035            |0.178559864855353    |
|30D               |90D+                   |166             |71035            |0.0023368761877947493|
|60D               |90D+                   |7054            |15149            |0.4656412964552116   |
|60D               |60D                    |3664            |15149            |0.2418641494488085   |
|60D               |Current                |2767            |15149            |0.1

In [0]:
# Verifying that the transition rates and counts reconcile correctly within each starting bucket

transition_rate_validation = (transition_matrix_summary.groupBy("delinquency_bucket")
                                                        .agg(spark_sum("transition_rate").alias("transition_rate_sum"),
                                                             spark_sum("transition_count").alias("transition_count_sum"),
                                                             spark_max("from_bucket_total").alias("from_bucket_total")
                                                        )
                                                        .withColumn("rate_difference_from_1", spark_abs(col("transition_rate_sum") - lit(1.0)))
                                                        .withColumn("count_reconciliation_diff", col("transition_count_sum") - col("from_bucket_total"))
                                                        .orderBy("delinquency_bucket")
)

print("Transition reconciliation check")
transition_rate_validation.show(truncate = False)

Transition reconciliation check
+------------------+-------------------+--------------------+-----------------+----------------------+-------------------------+
|delinquency_bucket|transition_rate_sum|transition_count_sum|from_bucket_total|rate_difference_from_1|count_reconciliation_diff|
+------------------+-------------------+--------------------+-----------------+----------------------+-------------------------+
|30D               |1.0                |71035               |71035            |0.0                   |0                        |
|60D               |1.0                |15149               |15149            |0.0                   |0                        |
|90D+              |1.0                |22408               |22408            |0.0                   |0                        |
|Current           |0.9999999999999999 |12976503            |12976503         |1.1102230246251565E-16|0                        |
+------------------+-------------------+--------------------+----

In [0]:
# Converting the transition matrix into headline roll-rate and cure-rate metrics for each delinquency bucket
key_roll_rate_summary = (loan_transitions_df.groupBy("delinquency_bucket")
                                            .agg(count("*").alias("loan_months"),
                                                 avg(when(col("next_is_delinquent") == 1, 1)
                                                     .otherwise(0)).alias("next_month_delinquency_entry_rate"),
                                                 avg(when(col("next_delinquency_bucket") == "Current", 1)
                                                     .otherwise(0)).alias("next_month_cure_to_current_rate"),
                                                 avg(when(col("next_is_serious_delinquent") == 1, 1)
                                                     .otherwise(0)).alias("next_month_serious_delinquency_rate")
                                            ).orderBy("delinquency_bucket")
)

key_roll_rate_summary.show(truncate = False)

+------------------+-----------+---------------------------------+-------------------------------+-----------------------------------+
|delinquency_bucket|loan_months|next_month_delinquency_entry_rate|next_month_cure_to_current_rate|next_month_serious_delinquency_rate|
+------------------+-----------+---------------------------------+-------------------------------+-----------------------------------+
|30D               |71035      |0.4996973323009784               |0.5003026676990217             |0.0023368761877947493              |
|60D               |15149      |0.8173476797148327               |0.18265232028516734            |0.4656412964552116                 |
|90D+              |22408      |0.9034719742948947               |0.09652802570510532            |0.8671010353445198                 |
|Current           |12976503   |0.004238352967667791             |0.9957616470323322             |9.016296609340745E-6               |
+------------------+-----------+-----------------------

In [0]:
# How often currently performing loans are becoming delinquent next month across credit score bands
credit_roll_rate_summary = (loan_transitions_df.filter(col("delinquency_bucket") == "Current")
                                                .groupBy("credit_score_band")
                                                .agg(count("*").alias("current_loan_months"),
                                                     avg(when(col("next_is_delinquent") == 1, 1)
                                                         .otherwise(0)).alias("current_to_delinquent_next_month_rate"),
                                                     avg(when(col("next_delinquency_bucket") == "90D+", 1)
                                                         .otherwise(0)).alias("current_to_90_plus_next_month_rate")
                                                )
                                                .orderBy("credit_score_band")
)

credit_roll_rate_summary.show(truncate = False)

+-----------------+-------------------+-------------------------------------+----------------------------------+
|credit_score_band|current_loan_months|current_to_delinquent_next_month_rate|current_to_90_plus_next_month_rate|
+-----------------+-------------------+-------------------------------------+----------------------------------+
|620-679          |1021849            |0.014604897592501435                 |3.03371633186508E-5               |
|680-739          |3342136            |0.0063399574403914145                |1.2866023405391044E-5             |
|740-799          |6932477            |0.0023081216136743043                |4.615954730177972E-6              |
|800+             |1653665            |0.0014537406306597768                |5.442456603967551E-6              |
|<620             |22060              |0.020670897552130554                 |4.533091568449683E-5              |
|Missing          |4316               |0.0057924003707136235                |2.3169601482854495E

In [0]:
# How next-month delinquency entry changes across loan-to-value bands

ltv_roll_rate_summary = (loan_transitions_df.filter(col("delinquency_bucket") == "Current")
                                            .groupBy("ltv_band")
                                            .agg(count("*").alias("current_loan_months"),
                                                 avg(when(col("next_is_delinquent") == 1, 1)
                                                     .otherwise(0)).alias("current_to_delinquent_next_month_rate"),
                                                 avg(when(col("next_delinquency_bucket") == "90D+", 1)
                                                     .otherwise(0)).alias("current_to_90_plus_next_month_rate")
                                            )
                                            .filter(col("current_loan_months") >= 1000)
                                            .orderBy("ltv_band")
)

ltv_roll_rate_summary.show(truncate = False)

+--------+-------------------+-------------------------------------+----------------------------------+
|ltv_band|current_loan_months|current_to_delinquent_next_month_rate|current_to_90_plus_next_month_rate|
+--------+-------------------+-------------------------------------+----------------------------------+
|60-79   |3198708            |0.0041391711903681115                |7.190403125261825E-6              |
|80-89   |3189145            |0.0035081503036080205                |5.644146001514513E-6              |
|90-100  |4071979            |0.005139761280694228                 |1.399810755409102E-5              |
|<60     |2516334            |0.0038301751675254556                |7.550666962334889E-6              |
+--------+-------------------+-------------------------------------+----------------------------------+



In [0]:
# Measuring how next-month delinquency entry changes across debt-to-income bands

dti_roll_rate_summary = (loan_transitions_df.filter(col("delinquency_bucket") == "Current")
                                            .groupBy("dti_band")
                                            .agg(count("*").alias("current_loan_months"),
                                                 avg(when(col("next_is_delinquent") == 1, 1)
                                                     .otherwise(0)).alias("current_to_delinquent_next_month_rate"),
                                                 avg(when(col("next_delinquency_bucket") == "90D+", 1)
                                                     .otherwise(0)).alias("current_to_90_plus_next_month_rate")
                                            )
                                            .orderBy("dti_band")
)

dti_roll_rate_summary.show(truncate = False)

+--------+-------------------+-------------------------------------+----------------------------------+
|dti_band|current_loan_months|current_to_delinquent_next_month_rate|current_to_90_plus_next_month_rate|
+--------+-------------------+-------------------------------------+----------------------------------+
|20-29   |2006743            |0.002803547838462623                 |6.478158887311429E-6              |
|30-39   |3875617            |0.003612586073391669                 |7.224656099919058E-6              |
|40-49   |5788252            |0.005045219178432452                 |1.1402406115006741E-5             |
|50+     |750706             |0.006267433589181384                 |9.324555818123207E-6              |
|<20     |555031             |0.0026376905073770655                |5.405103498723495E-6              |
|Missing |154                |0.0                                  |0.0                               |
+--------+-------------------+----------------------------------

In [0]:
# Ranking 30-day delinquent segments by whether they are curing, staying stressed or rolling worse next month
transition_30d_segment_summary = (loan_transitions_df.filter(col("delinquency_bucket") == "30D")
                                                        .groupBy("credit_score_band", "ltv_band", "dti_band")
                                                        .agg(count("*").alias("loan_months_30d"),
                                                             avg(when(col("next_delinquency_bucket") == "Current", 1)
                                                                 .otherwise(0)).alias("cure_to_current_rate"),
                                                             avg(when(col("next_delinquency_bucket") == "30D", 1)
                                                                 .otherwise(0)).alias("stay_30d_rate"),
                                                             avg(when(col("next_delinquency_bucket") == "60D", 1)
                                                                 .otherwise(0)).alias("roll_to_60d_rate"),
                                                             avg(when(col("next_delinquency_bucket") == "90D+", 1)
                                                                 .otherwise(0)).alias("roll_to_90d_plus_rate")
                                                        )
                                                        .filter(col("loan_months_30d") >= 100)
                                                        .orderBy(col("roll_to_60d_rate").desc(), 
                                                                 col("loan_months_30d").desc())
)

transition_30d_segment_summary.show(30, truncate = False)

+-----------------+--------+--------+---------------+--------------------+-------------------+-------------------+---------------------+
|credit_score_band|ltv_band|dti_band|loan_months_30d|cure_to_current_rate|stay_30d_rate      |roll_to_60d_rate   |roll_to_90d_plus_rate|
+-----------------+--------+--------+---------------+--------------------+-------------------+-------------------+---------------------+
|620-679          |<60     |<20     |149            |0.46308724832214765 |0.2953020134228188 |0.24161073825503357|0.0                  |
|740-799          |90-100  |50+     |729            |0.4609053497942387  |0.30315500685871055|0.2345679012345679 |0.0013717421124828531|
|620-679          |90-100  |20-29   |568            |0.3485915492957746  |0.41901408450704225|0.22887323943661972|0.0035211267605633804|
|620-679          |60-79   |50+     |364            |0.43131868131868134 |0.34065934065934067|0.22802197802197802|0.0                  |
|620-679          |90-100  |40-49   |4607

In [0]:
# Ranking 60-day delinquent segments by whether they are improving or rolling into 90D+ distress next month

transition_60d_segment_summary = (loan_transitions_df.filter(col("delinquency_bucket") == "60D")
                                                        .groupBy("credit_score_band", "ltv_band", "dti_band")
                                                        .agg(count("*").alias("loan_months_60d"),
                                                             avg(when(col("next_delinquency_bucket") == "Current", 1)
                                                                 .otherwise(0)).alias("cure_to_current_rate"),
                                                             avg(when(col("next_delinquency_bucket") == "30D", 1)
                                                                 .otherwise(0)).alias("improve_to_30d_rate"),
                                                             avg(when(col("next_delinquency_bucket") == "60D", 1)
                                                                 .otherwise(0)).alias("stay_60d_rate"),
                                                             avg(when(col("next_delinquency_bucket") == "90D+", 1)
                                                                 .otherwise(0)).alias("roll_to_90d_plus_rate")
                                                        )
                                                        .filter(col("loan_months_60d") >= 50)
                                                        .orderBy(col("roll_to_90d_plus_rate").desc(), 
                                                                 col("loan_months_60d").desc())
)

transition_60d_segment_summary.show(30, truncate = False)

+-----------------+--------+--------+---------------+--------------------+--------------------+-------------------+---------------------+
|credit_score_band|ltv_band|dti_band|loan_months_60d|cure_to_current_rate|improve_to_30d_rate |stay_60d_rate      |roll_to_90d_plus_rate|
+-----------------+--------+--------+---------------+--------------------+--------------------+-------------------+---------------------+
|740-799          |90-100  |50+     |175            |0.18857142857142858 |0.08571428571428572 |0.11428571428571428|0.6114285714285714   |
|620-679          |90-100  |50+     |133            |0.09774436090225563 |0.08270676691729323 |0.21052631578947367|0.6090225563909775   |
|800+             |90-100  |40-49   |56             |0.23214285714285715 |0.07142857142857142 |0.08928571428571429|0.6071428571428571   |
|740-799          |90-100  |20-29   |69             |0.2608695652173913  |0.08695652173913043 |0.07246376811594203|0.5797101449275363   |
|680-739          |80-89   |50+   

In [0]:
# Ranking segments where already-distressed loans are most likely to worsen into 90D+ rather than cure

distress_persistence_watchlist = (transition_60d_segment_summary.filter(col("loan_months_60d") >= 100)
                                                                .select("credit_score_band", "ltv_band", "dti_band",
                                                                        "loan_months_60d", "cure_to_current_rate",
                                                                        "improve_to_30d_rate", "stay_60d_rate",
                                                                        "roll_to_90d_plus_rate"
                                                                )
                                                                .orderBy(col("roll_to_90d_plus_rate").desc(),
                                                                         col("cure_to_current_rate").asc(),
                                                                         col("loan_months_60d").desc()
                                                                )
)

distress_persistence_watchlist.show(30, truncate = False)

+-----------------+--------+--------+---------------+--------------------+--------------------+-------------------+---------------------+
|credit_score_band|ltv_band|dti_band|loan_months_60d|cure_to_current_rate|improve_to_30d_rate |stay_60d_rate      |roll_to_90d_plus_rate|
+-----------------+--------+--------+---------------+--------------------+--------------------+-------------------+---------------------+
|740-799          |90-100  |50+     |175            |0.18857142857142858 |0.08571428571428572 |0.11428571428571428|0.6114285714285714   |
|620-679          |90-100  |50+     |133            |0.09774436090225563 |0.08270676691729323 |0.21052631578947367|0.6090225563909775   |
|680-739          |80-89   |50+     |122            |0.1885245901639344  |0.06557377049180328 |0.1721311475409836 |0.5737704918032787   |
|740-799          |90-100  |30-39   |302            |0.2152317880794702  |0.056291390728476824|0.16556291390728478|0.5629139072847682   |
|740-799          |90-100  |40-49 

In [0]:
# Comparable month-3 and month-9 loan snapshots at the same seasoning point

month_3_status_window = (Window.partitionBy("loan_sequence_number")
                                .orderBy(col("loan_age").desc(), 
                                         col("monthly_reporting_period").desc())
)

month_3_status_df = (analysis_df.filter((col("loan_age") <= 3) & 
                                        ((col("loan_age") == 3) | (col("is_zero_balance") == 1)))
                                .withColumn("row_num", row_number().over(month_3_status_window))
                                .filter(col("row_num") == 1)
                                .drop("row_num")
)

month_9_status_window = (Window.partitionBy("loan_sequence_number")
                                .orderBy(col("loan_age").desc(), 
                                         col("monthly_reporting_period").desc())
)

month_9_status_df = (analysis_df.filter((col("loan_age") <= 9) & 
                                        ((col("loan_age") == 9) | (col("is_zero_balance") == 1)))
                                .withColumn("row_num", row_number().over(month_9_status_window))
                                .filter(col("row_num") == 1)
                                .drop("row_num")
)

print("Month 3 loans : ", month_3_status_df.count())
print("Month 6 loans : ", month_6_status_df.count())
print("Month 9 loans : ", month_9_status_df.count())

Month 3 loans :  1047695
Month 6 loans :  1047937
Month 9 loans :  966923


In [0]:
# Counting loans by origination quarter to establish the vintage base size before comparing performance

origination_counts_by_quarter = (latest_loan_df.groupBy("origination_quarter")
                                                .agg(count("*").alias("total_loans_originated"))
)

month_3_vintage_summary = (month_3_status_df.groupBy("origination_quarter")
                                            .agg(count("*").alias("loan_count_observed_by_month_3"),
                                                 avg("is_delinquent").alias("month_3_delinquency_rate"),
                                                 avg("is_serious_delinquent").alias("month_3_serious_delinquency_rate"),
                                                 avg("is_prepay").alias("prepay_by_month_3_share"),
                                                 avg("is_zero_balance").alias("terminated_by_month_3_share")
                                            )
)

month_9_vintage_summary = (month_9_status_df.groupBy("origination_quarter")
                                            .agg(count("*").alias("loan_count_observed_by_month_9"),
                                                 avg("is_delinquent").alias("month_9_delinquency_rate"),
                                                 avg("is_serious_delinquent").alias("month_9_serious_delinquency_rate"),
                                                 avg("is_prepay").alias("prepay_by_month_9_share"),
                                                 avg("is_zero_balance").alias("terminated_by_month_9_share")
                                            )
)

seasoning_vintage_summary = (origination_counts_by_quarter.join(month_3_vintage_summary, 
                                                                on = "origination_quarter", 
                                                                how = "left")
                                                          .join(month_6_vintage_summary, 
                                                                on = "origination_quarter", 
                                                                how = "left")
                                                          .join(month_9_vintage_summary, 
                                                                on = "origination_quarter", 
                                                                how = "left")
                                                          .withColumn("month_3_observation_coverage",
                                                                      col("loan_count_observed_by_month_3") / col("total_loans_originated"))
                                                          .withColumn("month_6_observation_coverage",
                                                                      col("loan_count_observed_by_month_6") / col("total_loans_originated"))
                                                          .withColumn("month_9_observation_coverage",
                                                                      col("loan_count_observed_by_month_9") / col("total_loans_originated"))
                                                          .orderBy("origination_quarter")
)

seasoning_vintage_summary.show(truncate = False)

+-------------------+----------------------+------------------------------+------------------------+--------------------------------+-----------------------+---------------------------+------------------------------+------------------------+--------------------------------+-----------------------+---------------------------+------------------------------+------------------------+--------------------------------+-----------------------+---------------------------+----------------------------+----------------------------+----------------------------+
|origination_quarter|total_loans_originated|loan_count_observed_by_month_3|month_3_delinquency_rate|month_3_serious_delinquency_rate|prepay_by_month_3_share|terminated_by_month_3_share|loan_count_observed_by_month_6|month_6_delinquency_rate|month_6_serious_delinquency_rate|prepay_by_month_6_share|terminated_by_month_6_share|loan_count_observed_by_month_9|month_9_delinquency_rate|month_9_serious_delinquency_rate|prepay_by_month_9_share|termin

In [0]:
# Measuring coverage at month 3, 6 and 9 
# - later vintage comparisons can be made only where observation depth is sufficient

vintage_coverage_summary = (latest_loan_df.groupBy("origination_quarter")
                                            .agg(count("*").alias("latest_loan_count"))
                                            .join(month_3_vintage_summary
                                                  .select("origination_quarter", "loan_count_observed_by_month_3"),
                                                  on = "origination_quarter",
                                                  how = "left"
                                            )
                                            .join(month_6_vintage_summary
                                                  .select("origination_quarter", "loan_count_observed_by_month_6"),
                                                  on = "origination_quarter",
                                                  how = "left"
                                            )
                                            .join(month_9_vintage_summary
                                                  .select("origination_quarter", "loan_count_observed_by_month_9"),
                                                  on = "origination_quarter",
                                                  how = "left"
                                            )
                                            .withColumn("month_3_coverage_ratio", 
                                                        col("loan_count_observed_by_month_3") / col("latest_loan_count")
                                            )
                                            .withColumn("month_6_coverage_ratio",
                                                        col("loan_count_observed_by_month_6") / col("latest_loan_count")
                                            )
                                            .withColumn("month_9_coverage_ratio",
                                                        col("loan_count_observed_by_month_9") / col("latest_loan_count")
                                            )
                                            .orderBy("origination_quarter")
)

vintage_coverage_summary.show(truncate = False)

+-------------------+-----------------+------------------------------+------------------------------+------------------------------+----------------------+----------------------+----------------------+
|origination_quarter|latest_loan_count|loan_count_observed_by_month_3|loan_count_observed_by_month_6|loan_count_observed_by_month_9|month_3_coverage_ratio|month_6_coverage_ratio|month_9_coverage_ratio|
+-------------------+-----------------+------------------------------+------------------------------+------------------------------+----------------------+----------------------+----------------------+
|2024Q1             |215022           |214928                        |214965                        |214910                        |0.9995628354307932    |0.999734910846332     |0.9994791230664769    |
|2024Q2             |255625           |255488                        |255487                        |255381                        |0.9994640586797066    |0.9994601466992665    |0.999045476772

In [0]:
seasoning_vintage_comparable_summary = (seasoning_vintage_summary.join(vintage_coverage_summary
                                                                        .select("origination_quarter",
                                                                               "month_3_coverage_ratio",
                                                                               "month_6_coverage_ratio",
                                                                               "month_9_coverage_ratio"
                                                                        ),
                                                                        on = "origination_quarter",
                                                                        how = "inner"
                                                                )
                                                                .join(month_9_vintage_summary
                                                                      .select("origination_quarter"),
                                                                      on = "origination_quarter",
                                                                      how = "inner"
                                                                )
                                                                .orderBy("origination_quarter")
)

seasoning_vintage_comparable_summary.show(truncate = False)

+-------------------+----------------------+------------------------------+------------------------+--------------------------------+-----------------------+---------------------------+------------------------------+------------------------+--------------------------------+-----------------------+---------------------------+------------------------------+------------------------+--------------------------------+-----------------------+---------------------------+----------------------------+----------------------------+----------------------------+----------------------+----------------------+----------------------+
|origination_quarter|total_loans_originated|loan_count_observed_by_month_3|month_3_delinquency_rate|month_3_serious_delinquency_rate|prepay_by_month_3_share|terminated_by_month_3_share|loan_count_observed_by_month_6|month_6_delinquency_rate|month_6_serious_delinquency_rate|prepay_by_month_6_share|terminated_by_month_6_share|loan_count_observed_by_month_9|month_9_delinquency

In [0]:
# Keeping only vintages with strong month-9 coverage so longer-horizon comparisons are remaining fair

month_9_vintage_comparable_summary = (seasoning_vintage_comparable_summary.filter(col("month_9_coverage_ratio") >= 0.90)
                                                                            .select("origination_quarter",
                                                                                    "loan_count_observed_by_month_9",
                                                                                    "month_9_coverage_ratio",
                                                                                    "month_9_delinquency_rate",
                                                                                    "month_9_serious_delinquency_rate",
                                                                                    "prepay_by_month_9_share",
                                                                                    "terminated_by_month_9_share"
                                                                            )
                                                                            .orderBy("origination_quarter")
)

month_9_vintage_comparable_summary.show(truncate = False)

+-------------------+------------------------------+----------------------+------------------------+--------------------------------+-----------------------+---------------------------+
|origination_quarter|loan_count_observed_by_month_9|month_9_coverage_ratio|month_9_delinquency_rate|month_9_serious_delinquency_rate|prepay_by_month_9_share|terminated_by_month_9_share|
+-------------------+------------------------------+----------------------+------------------------+--------------------------------+-----------------------+---------------------------+
|2024Q1             |214910                        |0.9994791230664769    |0.011218649667302592    |0.002736029035410172            |0.05793122702526639    |0.06086733981666744        |
|2024Q2             |255381                        |0.9990454767726161    |0.010533281645854625    |0.0028858842278791296           |0.062103288811618716   |0.06489911152356675        |
|2024Q3             |295794                        |0.9992534162120162

In [0]:
# Reusable logistic modeling function : estimating baseline-relative driver relationships for delinquency and prepayment

def fit_logistic_driver_model(input_df, target_col, categorical_cols, model_name) : 
    model_df = (input_df.select([target_col] + categorical_cols)
                        .fillna("Missing", subset = categorical_cols)
    )

    # Splitting the sample so model relationships can be checked on unseen data, not only on the training data
    train_df, test_df = model_df.randomSplit([0.8, 0.2], seed = 42)

    # Encoding categorical features so the logistic model can use borrower and loan characteristics numerically
    indexers = [StringIndexer(inputCol = c, outputCol = f"{c}_idx", handleInvalid = "keep")
                for c in categorical_cols
    ]

    encoders = [OneHotEncoder(inputCol = f"{c}_idx", outputCol = f"{c}_ohe")
                for c in categorical_cols
    ]

    assembler = VectorAssembler(inputCols = [f"{c}_ohe" for c in categorical_cols],
                                outputCol = "features"
    )

    lr = LogisticRegression(featuresCol = "features",
                            labelCol = target_col,
                            maxIter = 30,
                            regParam = 0.05,
                            elasticNetParam = 0.0
    )

    pipeline = Pipeline(stages = indexers + encoders + [assembler, lr])
    pipeline_model = pipeline.fit(train_df)

    train_scored_df = pipeline_model.transform(train_df)
    test_scored_df = pipeline_model.transform(test_df)

    feature_metadata = train_scored_df.schema["features"].metadata["ml_attr"]["attrs"]
    feature_names = []
    for attr_group in feature_metadata.values():
        feature_names.extend([item["name"] for item in attr_group])

    lr_model = pipeline_model.stages[-1]

    # Converting fitted coefficients into odds ratios so the business effect size is easier to interpret
    coefficient_rows = []
    for feature_name, coefficient in zip(feature_names, lr_model.coefficients) : 
        coefficient_rows.append(
            {
                "model_name" : model_name,
                "feature" : feature_name,
                "coefficient" : float(coefficient),
                "odds_ratio" : float(exp(coefficient)),
            }
        )

    coefficient_df = (
        pd.DataFrame(coefficient_rows)
        .sort_values("odds_ratio", ascending = False)
        .reset_index(drop = True)
    )

    return pipeline_model, train_scored_df, test_scored_df, coefficient_df

In [0]:
# Estimating which borrower and loan traits are associated with 'active delinquency' 
# - Comparing categories against their baseline groups

driver_features = ["credit_score_band", "ltv_band", "dti_band", "loan_purpose", "occupancy_status",
                   "first_time_homebuyer_flag", "channel", "property_type",
]

BASELINE_ASSOCIATION_NOTE = ("Interpret these coefficients as baseline-relative associations from a regularized "
                             "logistic model. They are descriptive, not causal and each odds ratio is relative "
                             "to the omitted reference category within each encoded feature."
)

delinquency_model, delinquency_train_df, delinquency_test_df, delinquency_coefficients = fit_logistic_driver_model(
    active_latest_loan_df,
    "is_delinquent",
    driver_features,
    "active_delinquency"
)

print(BASELINE_ASSOCIATION_NOTE)
print("Largest positive baseline-relative associations with active delinquency")
display(delinquency_coefficients.head(20))

print("Largest negative / protective baseline-relative associations with active delinquency")
display(delinquency_coefficients.sort_values("odds_ratio", ascending = True).head(20))

Interpret these coefficients as baseline-relative associations from a regularized logistic model. They are descriptive, not causal and each odds ratio is relative to the omitted reference category within each encoded feature.
Largest positive baseline-relative associations with active delinquency


model_name,feature,coefficient,odds_ratio
active_delinquency,credit_score_band_ohe_<620,1.006190854508623,2.735162514794001
active_delinquency,credit_score_band_ohe_620-679,0.6879532034984689,1.9896389766286862
active_delinquency,property_type_ohe_MH,0.18189011943977865,1.1994823868418436
active_delinquency,credit_score_band_ohe_Missing,0.17015176086405795,1.1854847478590558
active_delinquency,credit_score_band_ohe_680-739,0.1493393889647824,1.1610669756668903
active_delinquency,dti_band_ohe_50+,0.14291359974664475,1.1536301234250177
active_delinquency,ltv_band_ohe_90-100,0.13838355456171947,1.1484159459857217
active_delinquency,dti_band_ohe_40-49,0.08677020160698615,1.090646022215783
active_delinquency,loan_purpose_ohe_C,0.08171426590841868,1.0851457024060172
active_delinquency,occupancy_status_ohe_P,0.06538902026289736,1.0675742516290812


Largest negative / protective baseline-relative associations with active delinquency


model_name,feature,coefficient,odds_ratio
active_delinquency,ltv_band_ohe_100+,-0.24305871545238814,0.7842254662666935
active_delinquency,credit_score_band_ohe_740-799,-0.23912306195173566,0.7873179875225944
active_delinquency,dti_band_ohe_Missing,-0.23689562869051356,0.7890736403690501
active_delinquency,ltv_band_ohe_Missing,-0.20374943093920717,0.8156667264292343
active_delinquency,credit_score_band_ohe_800+,-0.19390561448878788,0.8237356493098802
active_delinquency,dti_band_ohe_<20,-0.11044901087352516,0.8954319863202931
active_delinquency,dti_band_ohe_20-29,-0.10129332257958831,0.9036679277994376
active_delinquency,occupancy_status_ohe_S,-0.07821252952724601,0.9247678651795334
active_delinquency,property_type_ohe_CP,-0.07603862764911903,0.9267804065195976
active_delinquency,ltv_band_ohe_<60,-0.07451271338547526,0.9281956734745774


In [0]:
# Estimating which borrower and loan traits are associated with 'serious delinquency', not just early-stage delinquency

serious_delinquency_model, serious_delinquency_train_df, serious_delinquency_test_df, serious_delinquency_coefficients = fit_logistic_driver_model(
    active_latest_loan_df,
    "is_serious_delinquent",
    driver_features,
    "active_serious_delinquency"
)

print(BASELINE_ASSOCIATION_NOTE)
print("Largest positive baseline-relative associations with serious delinquency")
display(serious_delinquency_coefficients.head(20))

print("Largest negative / protective baseline-relative associations with serious delinquency")
display(serious_delinquency_coefficients.sort_values("odds_ratio", ascending = True).head(20))

Interpret these coefficients as baseline-relative associations from a regularized logistic model. They are descriptive, not causal and each odds ratio is relative to the omitted reference category within each encoded feature.
Largest positive baseline-relative associations with serious delinquency


model_name,feature,coefficient,odds_ratio
active_serious_delinquency,credit_score_band_ohe_<620,0.35264232346008056,1.4228221423093659
active_serious_delinquency,credit_score_band_ohe_620-679,0.27431926572933146,1.3156347722892436
active_serious_delinquency,ltv_band_ohe_90-100,0.07754841270827828,1.0806345476373
active_serious_delinquency,credit_score_band_ohe_Missing,0.07504620130120843,1.0779339516853763
active_serious_delinquency,dti_band_ohe_50+,0.07336979673431979,1.0761284121162036
active_serious_delinquency,credit_score_band_ohe_680-739,0.059459218194639445,1.0612624798971104
active_serious_delinquency,property_type_ohe_MH,0.05388782372476152,1.0553662084644444
active_serious_delinquency,occupancy_status_ohe_P,0.04640865903278394,1.0475023948167506
active_serious_delinquency,dti_band_ohe_40-49,0.038959964905833123,1.0397288571741985
active_serious_delinquency,first_time_homebuyer_flag_ohe_Y,0.026868952422356266,1.0272331775229366


Largest negative / protective baseline-relative associations with serious delinquency


model_name,feature,coefficient,odds_ratio
active_serious_delinquency,credit_score_band_ohe_740-799,-0.09739542477653568,0.9071972069460732
active_serious_delinquency,ltv_band_ohe_100+,-0.08072024377824999,0.9224517169578869
active_serious_delinquency,ltv_band_ohe_Missing,-0.07666911654549145,0.9261962659303397
active_serious_delinquency,dti_band_ohe_Missing,-0.0752491561050524,0.9275123621692049
active_serious_delinquency,credit_score_band_ohe_800+,-0.07197245697277453,0.9305565258019654
active_serious_delinquency,dti_band_ohe_<20,-0.04912931110379392,0.9520580100661511
active_serious_delinquency,occupancy_status_ohe_S,-0.0480929600136262,0.9530451878658261
active_serious_delinquency,dti_band_ohe_20-29,-0.04709114754671227,0.9540004388278308
active_serious_delinquency,property_type_ohe_CP,-0.04683131924007874,0.9542483473518104
active_serious_delinquency,occupancy_status_ohe_I,-0.044105851642108215,0.956852667686166


In [0]:
# Estimating which borrower and loan traits are associated with 'early prepayment' within the first six months
prepay_driver_features = ["credit_score_band", "ltv_band", "dti_band", "loan_purpose",
                          "occupancy_status", "first_time_homebuyer_flag", "channel", "property_type",
]

month_6_prepay_model, month_6_prepay_train_df, month_6_prepay_test_df, month_6_prepay_coefficients = fit_logistic_driver_model(
    month_6_status_df,
    "is_prepay",
    prepay_driver_features,
    "month_6_prepay"
)

print(BASELINE_ASSOCIATION_NOTE)
print("Largest positive baseline-relative associations with month-6 prepayment")
display(month_6_prepay_coefficients.head(20))

print("Largest negative baseline-relative associations with month-6 prepayment")
display(month_6_prepay_coefficients.sort_values("odds_ratio", ascending = True).head(20))

Interpret these coefficients as baseline-relative associations from a regularized logistic model. They are descriptive, not causal and each odds ratio is relative to the omitted reference category within each encoded feature.
Largest positive baseline-relative associations with month-6 prepayment


model_name,feature,coefficient,odds_ratio
month_6_prepay,ltv_band_ohe_Missing,1.2953690747589108,3.652343713293165
month_6_prepay,credit_score_band_ohe_800+,0.3425879942228641,1.408588295841479
month_6_prepay,first_time_homebuyer_flag_ohe_N,0.2558131391499357,1.291511372477534
month_6_prepay,property_type_ohe_PU,0.13032276243440427,1.1391960136714336
month_6_prepay,ltv_band_ohe_<60,0.0924688825856533,1.096879008990637
month_6_prepay,dti_band_ohe_50+,0.05540050831196101,1.0569638527231167
month_6_prepay,loan_purpose_ohe_P,0.054424612240852016,1.0559328690002456
month_6_prepay,occupancy_status_ohe_S,0.052811640407047195,1.0542310518846614
month_6_prepay,dti_band_ohe_<20,0.04581504248110598,1.0468807645805178
month_6_prepay,ltv_band_ohe_60-79,0.04487911655965147,1.0459014201047039


Largest negative baseline-relative associations with month-6 prepayment


model_name,feature,coefficient,odds_ratio
month_6_prepay,dti_band_ohe_Missing,-0.4260597746205517,0.65307730350592
month_6_prepay,ltv_band_ohe_100+,-0.35596463533080286,0.7004973926557804
month_6_prepay,first_time_homebuyer_flag_ohe_Y,-0.2558131391499329,0.7742866391348001
month_6_prepay,property_type_ohe_CP,-0.17669474044899275,0.8380355638414141
month_6_prepay,credit_score_band_ohe_620-679,-0.17241593552004145,0.8416290369453379
month_6_prepay,credit_score_band_ohe_680-739,-0.16848019963037544,0.8449479935358901
month_6_prepay,credit_score_band_ohe_<620,-0.16353641724158285,0.8491355752631278
month_6_prepay,property_type_ohe_MH,-0.1349008199639354,0.8738025711610397
month_6_prepay,loan_purpose_ohe_N,-0.1320061508495806,0.8763356048543085
month_6_prepay,ltv_band_ohe_90-100,-0.11549864169484707,0.8909217823799042


In [0]:
# How long loans are taking to first become delinquent or prepaid
# - Building a loan-level timing table by matching each loan to its first observed delinquency and first observed prepayment month
loan_event_timing_df = (latest_loan_df.select("loan_sequence_number", "first_payment_date", "credit_score_band",
                                              "ltv_band", "dti_band", "loan_purpose", "occupancy_status"
                                    )
                                    .join(analysis_df.groupBy("loan_sequence_number")
                                                        .agg(spark_min(when(col("is_delinquent") == 1, col("monthly_reporting_period"))).alias("first_delinquent_period"),
                                                             spark_min(when(col("is_prepay") == 1, col("monthly_reporting_period"))).alias("first_prepay_period")
                                                        ),
                                                        on = "loan_sequence_number",
                                                        how = "inner"
    )
    .withColumn("months_to_first_delinquency",
                when(col("first_delinquent_period").isNotNull(),
                     floor(months_between(col("first_delinquent_period"), col("first_payment_date")))
        )
    )
    .withColumn("months_to_first_prepay",
                when(col("first_prepay_period").isNotNull(),
                     floor(months_between(col("first_prepay_period"), col("first_payment_date")))
        )
    )
)

print("Loans in event timing table : ", loan_event_timing_df.count())
loan_event_timing_df.select("loan_sequence_number", "first_payment_date", "first_delinquent_period",
                            "months_to_first_delinquency", "first_prepay_period", "months_to_first_prepay"
).show(10, truncate = False)

Loans in event timing table :  1048407
+--------------------+------------------+-----------------------+---------------------------+-------------------+----------------------+
|loan_sequence_number|first_payment_date|first_delinquent_period|months_to_first_delinquency|first_prepay_period|months_to_first_prepay|
+--------------------+------------------+-----------------------+---------------------------+-------------------+----------------------+
|F24Q10000006        |2024-03-01        |NULL                   |NULL                       |NULL               |NULL                  |
|F24Q10000044        |2024-03-01        |NULL                   |NULL                       |NULL               |NULL                  |
|F24Q10000050        |2024-03-01        |NULL                   |NULL                       |2025-07-01         |16                    |
|F24Q10000054        |2024-03-01        |NULL                   |NULL                       |NULL               |NULL                  |
|F

In [0]:
# Checking how much of the portfolio is actually contributing to the event-timing analysis
event_timing_coverage_summary = pd.DataFrame([
    {
        "latest_loans_in_event_table":int(loan_event_timing_df.count()),
        "share_with_any_observed_delinquency":float(
            loan_event_timing_df.filter(col("months_to_first_delinquency").isNotNull()).count()
        ) / float(loan_event_timing_df.count()),
        "share_with_any_observed_prepay":float(
            loan_event_timing_df.filter(col("months_to_first_prepay").isNotNull()).count()
        ) / float(loan_event_timing_df.count())
    }
])

print("Event-timing note : the ranking tables below are conditional on loans that actually experienced "
      "the event. They describe speed among event loans, not unconditional portfolio risk."
)
event_timing_coverage_summary

Event-timing note : the ranking tables below are conditional on loans that actually experienced the event. They describe speed among event loans, not unconditional portfolio risk.


,latest_loans_in_event_table,share_with_any_observed_delinquency,share_with_any_observed_prepay
0,1048407,0.04126,0.087595


In [0]:
# Ranking which segment types are reaching first delinquency faster once delinquency occurs.

time_to_delinquency_summary = (loan_event_timing_df.filter(col("months_to_first_delinquency").isNotNull())
                                                    .groupBy("credit_score_band", "ltv_band")
                                                    .agg(count("*").alias("loan_count_with_delinquency"),
                                                         percentile_approx("months_to_first_delinquency", 0.5).alias("median_months_to_first_delinquency"),
                                                         percentile_approx("months_to_first_delinquency", 0.75).alias("p75_months_to_first_delinquency")
                                                    )
                                                    .filter(col("loan_count_with_delinquency") >= 100)
                                                    .orderBy(col("median_months_to_first_delinquency").asc(), 
                                                             col("loan_count_with_delinquency").desc())
)

time_to_delinquency_summary.show(30, truncate = False)

+-----------------+--------+---------------------------+----------------------------------+-------------------------------+
|credit_score_band|ltv_band|loan_count_with_delinquency|median_months_to_first_delinquency|p75_months_to_first_delinquency|
+-----------------+--------+---------------------------+----------------------------------+-------------------------------+
|800+             |60-79   |588                        |2                                 |7                              |
|800+             |80-89   |512                        |2                                 |7                              |
|740-799          |60-79   |3555                       |3                                 |7                              |
|740-799          |<60     |2411                       |3                                 |8                              |
|800+             |<60     |608                        |3                                 |6                              |
|800+   

In [0]:
# Ranking which segment types are prepaying faster once prepayment occurs
time_to_prepay_summary = (loan_event_timing_df.filter(col("months_to_first_prepay").isNotNull())
                                                .groupBy("credit_score_band", "loan_purpose")
                                                .agg(count("*").alias("loan_count_with_prepay"),
                                                     percentile_approx("months_to_first_prepay", 0.5).alias("median_months_to_first_prepay"),
                                                     percentile_approx("months_to_first_prepay", 0.75).alias("p75_months_to_first_prepay")
                                                )
                                                .filter(col("loan_count_with_prepay") >= 100)
                                                .orderBy(col("median_months_to_first_prepay").asc(), 
                                                         col("loan_count_with_prepay").desc())
)

time_to_prepay_summary.show(30, truncate = False)

+-----------------+------------+----------------------+-----------------------------+--------------------------+
|credit_score_band|loan_purpose|loan_count_with_prepay|median_months_to_first_prepay|p75_months_to_first_prepay|
+-----------------+------------+----------------------+-----------------------------+--------------------------+
|800+             |P           |14912                 |6                            |9                         |
|740-799          |P           |37933                 |7                            |11                        |
|740-799          |N           |4474                  |7                            |10                        |
|800+             |C           |1636                  |7                            |10                        |
|800+             |N           |881                   |7                            |10                        |
|740-799          |C           |6236                  |8                            |11         

In [0]:
# Building an overall origination profile by quarter

overall_origination_profile = (latest_loan_df.agg(avg("credit_score_clean").alias("overall_avg_credit_score"),
                                                  avg("loan_to_value_clean").alias("overall_avg_ltv"),
                                                  avg("debt_to_income_ratio_clean").alias("overall_avg_dti"),
                                                  avg(when(col("credit_score_clean") < 680, 1)
                                                      .otherwise(0)).alias("overall_share_credit_below_680"),
                                                  avg(when(col("loan_to_value_clean") >= 90, 1)
                                                      .otherwise(0)).alias("overall_share_ltv_90_plus"),
                                                  avg(when(col("debt_to_income_ratio_clean") >= 45, 1)
                                                      .otherwise(0)).alias("overall_share_dti_45_plus"),
                                                  avg(when(col("first_time_homebuyer_flag") == "Y", 1)
                                                      .otherwise(0)).alias("overall_share_first_time_homebuyer")
                                            )
)

vintage_composition_delta_summary = (origination_profile_summary.crossJoin(overall_origination_profile)
                                                                .withColumn("avg_credit_score_delta", col("avg_credit_score") - col("overall_avg_credit_score"))
                                                                .withColumn("avg_ltv_delta", col("avg_ltv") - col("overall_avg_ltv"))
                                                                .withColumn("avg_dti_delta", col("avg_dti") - col("overall_avg_dti"))
                                                                .withColumn("share_credit_below_680_delta", col("share_credit_below_680") - col("overall_share_credit_below_680"))
                                                                .withColumn("share_ltv_90_plus_delta", col("share_ltv_90_plus") - col("overall_share_ltv_90_plus"))
                                                                .withColumn("share_dti_45_plus_delta", col("share_dti_45_plus") - col("overall_share_dti_45_plus"))
                                                                .withColumn("share_first_time_homebuyer_delta", col("share_first_time_homebuyer") - col("overall_share_first_time_homebuyer"))
                                                                .select("origination_quarter", "avg_credit_score_delta",
                                                                        "avg_ltv_delta", "avg_dti_delta",
                                                                        "share_credit_below_680_delta",
                                                                        "share_ltv_90_plus_delta",
                                                                        "share_dti_45_plus_delta",
                                                                        "share_first_time_homebuyer_delta"
                                                                )
                                                                .orderBy("origination_quarter")
)

vintage_composition_delta_summary.show(truncate = False)

+-------------------+----------------------+-------------------+--------------------+----------------------------+-----------------------+-----------------------+--------------------------------+
|origination_quarter|avg_credit_score_delta|avg_ltv_delta      |avg_dti_delta       |share_credit_below_680_delta|share_ltv_90_plus_delta|share_dti_45_plus_delta|share_first_time_homebuyer_delta|
+-------------------+----------------------+-------------------+--------------------+----------------------------+-----------------------+-----------------------+--------------------------------+
|2024Q1             |-1.0801752679732317   |0.4638441682964469 |0.10062336732585209 |-1.3245370406549106E-4      |0.013690803818797892   |0.006220925042624725   |0.015283910725608585            |
|2024Q2             |-1.1290826788277855   |0.27333222624864106|0.1925665495335025  |0.006016540450146157        |0.012221551060923486   |0.011014389465290597   |0.03082688941074868             |
|2024Q3             

In [0]:
# Comparing vintage performance with changes in borrower mix.
# Showing whether quarter-to-quarter risk differences are partly explained by composition changes.
vintage_explanation_summary = (
    seasoning_vintage_summary
    .join(vintage_composition_delta_summary, on = "origination_quarter", how = "inner")
    .select(
        "origination_quarter",
        "month_3_delinquency_rate",
        "month_6_delinquency_rate",
        "month_9_delinquency_rate",
        "prepay_by_month_6_share",
        "prepay_by_month_9_share",
        "avg_credit_score_delta",
        "avg_ltv_delta",
        "avg_dti_delta",
        "share_credit_below_680_delta",
        "share_ltv_90_plus_delta",
        "share_dti_45_plus_delta",
        "share_first_time_homebuyer_delta"
    )
    .orderBy("origination_quarter")
)

vintage_explanation_summary.show(truncate = False)

+-------------------+------------------------+------------------------+------------------------+-----------------------+-----------------------+----------------------+-------------------+--------------------+----------------------------+-----------------------+-----------------------+--------------------------------+
|origination_quarter|month_3_delinquency_rate|month_6_delinquency_rate|month_9_delinquency_rate|prepay_by_month_6_share|prepay_by_month_9_share|avg_credit_score_delta|avg_ltv_delta      |avg_dti_delta       |share_credit_below_680_delta|share_ltv_90_plus_delta|share_dti_45_plus_delta|share_first_time_homebuyer_delta|
+-------------------+------------------------+------------------------+------------------------+-----------------------+-----------------------+----------------------+-------------------+--------------------+----------------------------+-----------------------+-----------------------+--------------------------------+
|2024Q1             |0.0056670140698280354 

In [0]:
# Combining performance metrics with coverage checks so incomplete vintages are not over-interpreted.
vintage_presentation_summary = (vintage_explanation_summary.join(vintage_coverage_summary
                                                                 .select("origination_quarter",
                                                                         "month_3_coverage_ratio",
                                                                         "month_6_coverage_ratio",
                                                                         "month_9_coverage_ratio"
                                                                ),
                                                                on = "origination_quarter",
                                                                how = "left"
                                                            )
                                                            .withColumn("month_9_comparable",
                                                                        when(col("month_9_coverage_ratio") >= 0.90, "Yes")
                                                                        .otherwise("No")
                                                            )
                                                            .select("origination_quarter", "month_3_coverage_ratio",
                                                                    "month_6_coverage_ratio", "month_9_coverage_ratio",
                                                                    "month_9_comparable", "month_3_delinquency_rate",
                                                                    "month_6_delinquency_rate",
                                                                    "month_9_delinquency_rate",
                                                                    "prepay_by_month_6_share",
                                                                    "prepay_by_month_9_share",
                                                                    "avg_credit_score_delta", "avg_ltv_delta",
                                                                    "avg_dti_delta", "share_credit_below_680_delta",
                                                                    "share_ltv_90_plus_delta",
                                                                    "share_dti_45_plus_delta",
                                                                    "share_first_time_homebuyer_delta"
                                                            )
                                                            .orderBy("origination_quarter")
)

vintage_presentation_summary.show(truncate = False)

+-------------------+----------------------+----------------------+----------------------+------------------+------------------------+------------------------+------------------------+-----------------------+-----------------------+----------------------+-------------------+--------------------+----------------------------+-----------------------+-----------------------+--------------------------------+
|origination_quarter|month_3_coverage_ratio|month_6_coverage_ratio|month_9_coverage_ratio|month_9_comparable|month_3_delinquency_rate|month_6_delinquency_rate|month_9_delinquency_rate|prepay_by_month_6_share|prepay_by_month_9_share|avg_credit_score_delta|avg_ltv_delta      |avg_dti_delta       |share_credit_below_680_delta|share_ltv_90_plus_delta|share_dti_45_plus_delta|share_first_time_homebuyer_delta|
+-------------------+----------------------+----------------------+----------------------+------------------+------------------------+------------------------+------------------------+--

In [0]:
# Fitting a state-adjusted delinquency model
# - Estimating whether some states still show elevated delinquency after controlling for borrower and loan mix

# Including state along with borrower and loan attributes so state effects are measured after adjusting for mix
state_driver_features = ["credit_score_band", "ltv_band", "dti_band", "loan_purpose", "occupancy_status",
                         "first_time_homebuyer_flag", "channel", "property_type", "property_state"
]

state_adjusted_delinquency_model, state_adjusted_delinquency_train_df, state_adjusted_delinquency_test_df, state_adjusted_delinquency_coefficients = fit_logistic_driver_model(active_latest_loan_df, "is_delinquent", state_driver_features, "state_adjusted_active_delinquency")

# Isolating only the encoded state coefficients from the full model output
state_adjusted_state_effects = (state_adjusted_delinquency_coefficients[
        state_adjusted_delinquency_coefficients["feature"].str.startswith("property_state_ohe_")
    ]
    .copy()
)

# Converting model feature names into readable state labels
state_adjusted_state_effects["property_state"] = state_adjusted_state_effects["feature"].str.replace(
                                                                    "property_state_ohe_",
                                                                    "",
                                                                    regex = False
)

state_adjusted_state_effects = (state_adjusted_state_effects[["property_state", "coefficient", "odds_ratio"]]
                                                                    .sort_values("odds_ratio", ascending = False)
                                                                    .reset_index(drop = True)
)

print(BASELINE_ASSOCIATION_NOTE)
print("Highest positive baseline-relative state associations with active delinquency")
display(state_adjusted_state_effects.head(20))

print("Lowest baseline-relative state associations with active delinquency")
display(state_adjusted_state_effects.sort_values("odds_ratio", ascending = True).head(20))

Interpret these coefficients as baseline-relative associations from a regularized logistic model. They are descriptive, not causal and each odds ratio is relative to the omitted reference category within each encoded feature.
Highest positive baseline-relative state associations with active delinquency


property_state,coefficient,odds_ratio
VI,0.2795171039378567,1.3224910324059247
MS,0.09265790002123164,1.0970863578437131
DC,0.07189561921248858,1.0745431765468656
LA,0.0631343362291018,1.065169920531066
IN,0.05199785951993616,1.0533734877860013
FL,0.04910909128638515,1.0503349268785938
IL,0.03671734036998433,1.0373997483629145
WV,0.0366906500074111,1.037372060157003
IA,0.0248359704220448,1.0251469523105958
MI,0.02432026876812928,1.0246184186263438


Lowest baseline-relative state associations with active delinquency


property_state,coefficient,odds_ratio
GU,-0.23592278477764006,0.7898416593776281
PR,-0.1969135065066676,0.8212616640310282
AK,-0.13865892952610095,0.870524888166481
OR,-0.08174505826185198,0.9215068593125192
WA,-0.07242017068455847,0.9301399961357529
SD,-0.070726004750871,0.9317171432279773
VA,-0.056883303089655196,0.9447043070080194
DE,-0.053649919311624134,0.9477638422495833
NE,-0.05015874874256844,0.9510784300110902
ND,-0.04910663854729748,0.9520795958998743


In [0]:
# Checking which states remain elevated after accounting for borrower and loan composition
raw_state_risk_pd = state_risk_summary.select("property_state", "active_loan_count", 
                                              "active_delinquency_rate").toPandas()

adjusted_state_risk_pd = state_adjusted_state_effects.copy()

state_risk_comparison = (raw_state_risk_pd.merge(adjusted_state_risk_pd, 
                                                 on = "property_state", 
                                                 how = "inner")
                                          .sort_values(["odds_ratio", "active_delinquency_rate"], 
                                                       ascending = [False, False])
                                          .reset_index(drop = True)
)

state_risk_comparison.head(20)

,property_state,active_loan_count,active_delinquency_rate,coefficient,odds_ratio
0,IN,26922,0.017124,0.051998,1.053373
1,FL,74394,0.016520,0.049109,1.050335
2,IL,45172,0.016138,0.036717,1.037400
3,MI,36164,0.016093,0.024320,1.024618
4,NY,33478,0.015533,0.020239,1.020445
5,AL,12698,0.014254,0.019239,1.019425
6,CT,10952,0.015248,0.018851,1.019030
7,MN,21363,0.015400,0.014924,1.015036
8,TX,85561,0.014633,0.012795,1.012877
9,OH,43942,0.014587,0.009223,1.009265


In [0]:
# Evaluating model quality on train and test samples
# - Checking whether the delinquency and prepayment models are separating outcomes with reasonable consistency

# Calculating standard classification quality metrics for each scored dataset
def evaluate_binary_model(scored_df, label_col, model_name, sample_name):
    roc_evaluator = BinaryClassificationEvaluator(labelCol = label_col, 
                                                  rawPredictionCol = "rawPrediction",
                                                  metricName = "areaUnderROC"
    )
    pr_evaluator = BinaryClassificationEvaluator(labelCol = label_col,
                                                 rawPredictionCol = "rawPrediction",
                                                 metricName = "areaUnderPR"
    )

    return {"model_name":model_name, "sample_name":sample_name, 
            "auc_roc":roc_evaluator.evaluate(scored_df), "auc_pr":pr_evaluator.evaluate(scored_df),
    }

# Collecting all model evaluation results into one comparison table
model_quality_summary = pd.DataFrame([evaluate_binary_model(delinquency_train_df, "is_delinquent", "active_delinquency", "train"),
                                      evaluate_binary_model(delinquency_test_df, "is_delinquent", "active_delinquency", "test"),
                                      evaluate_binary_model(serious_delinquency_train_df, "is_serious_delinquent", "active_serious_delinquency", "train"),
                                      evaluate_binary_model(serious_delinquency_test_df, "is_serious_delinquent", "active_serious_delinquency", "test"),
                                      evaluate_binary_model(month_6_prepay_train_df, "is_prepay", "month_6_prepay", "train"),
                                      evaluate_binary_model(month_6_prepay_test_df, "is_prepay", "month_6_prepay", "test"),
                                      evaluate_binary_model(state_adjusted_delinquency_train_df, "is_delinquent", "state_adjusted_active_delinquency", "train"),
                                      evaluate_binary_model(state_adjusted_delinquency_test_df, "is_delinquent", "state_adjusted_active_delinquency", "test")
])

model_quality_summary

,model_name,sample_name,auc_roc,auc_pr
0,active_delinquency,train,0.795092,0.051934
1,active_delinquency,test,0.797896,0.053309
2,active_serious_delinquency,train,0.806069,0.017612
3,active_serious_delinquency,test,0.802617,0.018587
4,month_6_prepay,train,0.686766,0.063887
5,month_6_prepay,test,0.688614,0.063120
6,state_adjusted_active_delinquency,train,0.796053,0.052450
7,state_adjusted_active_delinquency,test,0.798894,0.054135


In [0]:
# Checking how many active loans exist in each major category before interpreting segment-level results
credit_band_support = (active_latest_loan_df.groupBy("credit_score_band")
                                            .agg(count("*").alias("active_loan_count"))
                                            .orderBy(col("active_loan_count").desc())
)

ltv_band_support = (active_latest_loan_df.groupBy("ltv_band")
                                         .agg(count("*").alias("active_loan_count"))
                                         .orderBy(col("active_loan_count").desc())
)

dti_band_support = (active_latest_loan_df.groupBy("dti_band")
                                         .agg(count("*").alias("active_loan_count"))
                                         .orderBy(col("active_loan_count").desc())
)

loan_purpose_support = (active_latest_loan_df.groupBy("loan_purpose")
                                             .agg(count("*").alias("active_loan_count"))
                                             .orderBy(col("active_loan_count").desc())
)

occupancy_support = (active_latest_loan_df.groupBy("occupancy_status")
                                          .agg(count("*").alias("active_loan_count"))
                                          .orderBy(col("active_loan_count").desc())
)

property_type_support = (active_latest_loan_df.groupBy("property_type")
                                              .agg(count("*").alias("active_loan_count"))
                                              .orderBy(col("active_loan_count").desc())
)

channel_support = (active_latest_loan_df.groupBy("channel")
                                        .agg(count("*").alias("active_loan_count"))
                                        .orderBy(col("active_loan_count").desc())
)

credit_band_support.show(truncate = False)
ltv_band_support.show(truncate = False)
dti_band_support.show(truncate = False)
loan_purpose_support.show(truncate = False)
occupancy_support.show(truncate = False)
property_type_support.show(truncate = False)
channel_support.show(truncate = False)

+-----------------+-----------------+
|credit_score_band|active_loan_count|
+-----------------+-----------------+
|740-799          |508964           |
|680-739          |245810           |
|800+             |120668           |
|620-679          |75836            |
|<620             |1701             |
|Missing          |320              |
+-----------------+-----------------+

+--------+-----------------+
|ltv_band|active_loan_count|
+--------+-----------------+
|90-100  |300639           |
|60-79   |235918           |
|80-89   |233553           |
|<60     |183162           |
|100+    |22               |
|Missing |5                |
+--------+-----------------+

+--------+-----------------+
|dti_band|active_loan_count|
+--------+-----------------+
|40-49   |425394           |
|30-39   |284982           |
|20-29   |147521           |
|50+     |55005            |
|<20     |40387            |
|Missing |10               |
+--------+-----------------+

+------------+-----------------+
|loa

In [0]:
# Filtering the active delinquency driver table to keep meaningful business signals and remove missing or less useful baseline noise

delinquency_driver_table = delinquency_coefficients.copy()

# Removing placeholder missing-category effects so the ranking stays business-focused
delinquency_driver_table = delinquency_driver_table[~delinquency_driver_table["feature"]
                                                        .str.contains("Missing", na = False)
].copy()

# Removing the 100+ LTV bucket from the final display because it is less central to the main comparison ladder
delinquency_driver_table = delinquency_driver_table[~delinquency_driver_table["feature"]
                                                        .str.contains("ltv_band_ohe_100\\+", regex = True, na = False)
].copy()

# Sorting the remaining features from strongest positive association to weakest
delinquency_driver_table = delinquency_driver_table.sort_values(["odds_ratio", "feature"],
                                                                ascending = [False, True]
).reset_index(drop = True)

print("Filtered active delinquency drivers")
display(delinquency_driver_table.head(20))

print("Filtered protective drivers")
display(delinquency_driver_table.sort_values("odds_ratio", ascending = True).head(20))

Filtered active delinquency drivers


model_name,feature,coefficient,odds_ratio
active_delinquency,credit_score_band_ohe_<620,1.006190854508623,2.735162514794001
active_delinquency,credit_score_band_ohe_620-679,0.6879532034984689,1.9896389766286862
active_delinquency,property_type_ohe_MH,0.18189011943977865,1.1994823868418436
active_delinquency,credit_score_band_ohe_680-739,0.1493393889647824,1.1610669756668903
active_delinquency,dti_band_ohe_50+,0.14291359974664475,1.1536301234250177
active_delinquency,ltv_band_ohe_90-100,0.13838355456171947,1.1484159459857217
active_delinquency,dti_band_ohe_40-49,0.08677020160698615,1.090646022215783
active_delinquency,loan_purpose_ohe_C,0.08171426590841868,1.0851457024060172
active_delinquency,occupancy_status_ohe_P,0.06538902026289736,1.0675742516290812
active_delinquency,property_type_ohe_SF,0.05494737031141765,1.056485010735191


Filtered protective drivers


model_name,feature,coefficient,odds_ratio
active_delinquency,credit_score_band_ohe_740-799,-0.23912306195173566,0.7873179875225944
active_delinquency,credit_score_band_ohe_800+,-0.19390561448878788,0.8237356493098802
active_delinquency,dti_band_ohe_<20,-0.11044901087352516,0.8954319863202931
active_delinquency,dti_band_ohe_20-29,-0.10129332257958831,0.9036679277994376
active_delinquency,occupancy_status_ohe_S,-0.07821252952724601,0.9247678651795334
active_delinquency,property_type_ohe_CP,-0.07603862764911903,0.9267804065195976
active_delinquency,ltv_band_ohe_<60,-0.07451271338547526,0.9281956734745774
active_delinquency,property_type_ohe_PU,-0.06075451851493885,0.9410542228119594
active_delinquency,occupancy_status_ohe_I,-0.0590613594066043,0.9426489270037407
active_delinquency,dti_band_ohe_30-39,-0.054614597497809995,0.9468499959999498


In [0]:
serious_delinquency_driver_table = serious_delinquency_coefficients.copy()

# Removing missing-category effects to avoid distracting from the main borrower and loan signals
serious_delinquency_driver_table = serious_delinquency_driver_table[
    ~serious_delinquency_driver_table["feature"].str.contains("Missing", na = False)
].copy()

# Removing the 100+ LTV bucket from the presentation-focused driver ranking
serious_delinquency_driver_table = serious_delinquency_driver_table[
    ~serious_delinquency_driver_table["feature"].str.contains("ltv_band_ohe_100\\+", regex = True, na = False)
].copy()

# Sorting the remaining features by strongest serious-delinquency association
serious_delinquency_driver_table = serious_delinquency_driver_table.sort_values(
    ["odds_ratio", "feature"],
    ascending = [False, True]
).reset_index(drop = True)

print("Filtered serious delinquency drivers")
display(serious_delinquency_driver_table.head(20))

print("Filtered protective drivers")
display(serious_delinquency_driver_table.sort_values("odds_ratio", ascending = True).head(20))

Filtered serious delinquency drivers


model_name,feature,coefficient,odds_ratio
active_serious_delinquency,credit_score_band_ohe_<620,0.35264232346008056,1.4228221423093659
active_serious_delinquency,credit_score_band_ohe_620-679,0.27431926572933146,1.3156347722892436
active_serious_delinquency,ltv_band_ohe_90-100,0.07754841270827828,1.0806345476373
active_serious_delinquency,dti_band_ohe_50+,0.07336979673431979,1.0761284121162036
active_serious_delinquency,credit_score_band_ohe_680-739,0.059459218194639445,1.0612624798971104
active_serious_delinquency,property_type_ohe_MH,0.05388782372476152,1.0553662084644444
active_serious_delinquency,occupancy_status_ohe_P,0.04640865903278394,1.0475023948167506
active_serious_delinquency,dti_band_ohe_40-49,0.038959964905833123,1.0397288571741985
active_serious_delinquency,first_time_homebuyer_flag_ohe_Y,0.026868952422356266,1.0272331775229366
active_serious_delinquency,loan_purpose_ohe_C,0.025492335023104946,1.0258200433533793


Filtered protective drivers


model_name,feature,coefficient,odds_ratio
active_serious_delinquency,credit_score_band_ohe_740-799,-0.09739542477653568,0.9071972069460732
active_serious_delinquency,credit_score_band_ohe_800+,-0.07197245697277453,0.9305565258019654
active_serious_delinquency,dti_band_ohe_<20,-0.04912931110379392,0.9520580100661511
active_serious_delinquency,occupancy_status_ohe_S,-0.0480929600136262,0.9530451878658261
active_serious_delinquency,dti_band_ohe_20-29,-0.04709114754671227,0.9540004388278308
active_serious_delinquency,property_type_ohe_CP,-0.04683131924007874,0.9542483473518104
active_serious_delinquency,occupancy_status_ohe_I,-0.044105851642108215,0.956852667686166
active_serious_delinquency,ltv_band_ohe_<60,-0.043821067132270054,0.9571252033092402
active_serious_delinquency,ltv_band_ohe_60-79,-0.02697475692496766,0.9733858124653119
active_serious_delinquency,first_time_homebuyer_flag_ohe_N,-0.026868952422347533,0.9734888065155783


In [0]:
# Filtering the early prepayment driver table to highlight the clearest business-facing prepayment signals

prepay_driver_table = month_6_prepay_coefficients.copy()

# Removing missing-category effects so the ranking stays easier to explain
prepay_driver_table = prepay_driver_table[~prepay_driver_table["feature"]
                                                .str.contains("Missing", na = False)
].copy()

# Removing the 100+ LTV bucket from the final presentation table
prepay_driver_table = prepay_driver_table[~prepay_driver_table["feature"]
                                                .str.contains("ltv_band_ohe_100\\+", regex = True, na = False)
].copy()

# Sorting the remaining features by strongest positive prepayment association
prepay_driver_table = prepay_driver_table.sort_values(["odds_ratio", "feature"],
                                                      ascending = [False, True]
).reset_index(drop = True)

print("Filtered month-6 prepayment drivers")
display(prepay_driver_table.head(20))

print("Filtered negative prepayment drivers")
display(prepay_driver_table.sort_values("odds_ratio", ascending = True).head(20))

Filtered month-6 prepayment drivers


model_name,feature,coefficient,odds_ratio
month_6_prepay,credit_score_band_ohe_800+,0.3425879942228641,1.408588295841479
month_6_prepay,first_time_homebuyer_flag_ohe_N,0.2558131391499357,1.291511372477534
month_6_prepay,property_type_ohe_PU,0.13032276243440427,1.1391960136714336
month_6_prepay,ltv_band_ohe_<60,0.0924688825856533,1.096879008990637
month_6_prepay,dti_band_ohe_50+,0.05540050831196101,1.0569638527231167
month_6_prepay,loan_purpose_ohe_P,0.054424612240852016,1.0559328690002456
month_6_prepay,occupancy_status_ohe_S,0.052811640407047195,1.0542310518846614
month_6_prepay,dti_band_ohe_<20,0.04581504248110598,1.0468807645805178
month_6_prepay,ltv_band_ohe_60-79,0.04487911655965147,1.0459014201047039
month_6_prepay,occupancy_status_ohe_P,0.03882358440361008,1.0395870680993335


Filtered negative prepayment drivers


model_name,feature,coefficient,odds_ratio
month_6_prepay,first_time_homebuyer_flag_ohe_Y,-0.2558131391499329,0.7742866391348001
month_6_prepay,property_type_ohe_CP,-0.17669474044899275,0.8380355638414141
month_6_prepay,credit_score_band_ohe_620-679,-0.17241593552004145,0.8416290369453379
month_6_prepay,credit_score_band_ohe_680-739,-0.16848019963037544,0.8449479935358901
month_6_prepay,credit_score_band_ohe_<620,-0.16353641724158285,0.8491355752631278
month_6_prepay,property_type_ohe_MH,-0.1349008199639354,0.8738025711610397
month_6_prepay,loan_purpose_ohe_N,-0.1320061508495806,0.8763356048543085
month_6_prepay,ltv_band_ohe_90-100,-0.11549864169484707,0.8909217823799042
month_6_prepay,property_type_ohe_SF,-0.1126498923097755,0.8934634137761873
month_6_prepay,occupancy_status_ohe_I,-0.06520776779306987,0.9368727911625885


In [0]:
# Keeping only state-adjusted effects with meaningful loan volume so the state comparison stays credible

# Keeping only states with enough active loans to avoid overreading very small populations
state_adjusted_state_effects_filtered = (state_risk_comparison.query("active_loan_count >= 10000")
                                                              .sort_values(["odds_ratio", "active_delinquency_rate"],
                                                                           ascending = [False, False])
                                                              .reset_index(drop = True)
)

print("Largest positive adjusted state effects with meaningful footprint")
display(state_adjusted_state_effects_filtered.head(20))

# Ranking states by adjusted delinquency association after borrower-mix controls
print("Lowest adjusted state effects with meaningful footprint")
display(
    state_adjusted_state_effects_filtered
    .sort_values(["odds_ratio", "active_delinquency_rate"], ascending = [True, True])
    .head(20)
)

Largest positive adjusted state effects with meaningful footprint


property_state,active_loan_count,active_delinquency_rate,coefficient,odds_ratio
IN,26922,0.017123542084540526,0.05199785951993616,1.0533734877860013
FL,74394,0.01652014947441998,0.04910909128638515,1.0503349268785938
IL,45172,0.01613831577083149,0.03671734036998433,1.0373997483629145
MI,36164,0.016093352505253844,0.02432026876812928,1.0246184186263438
NY,33478,0.015532588565625187,0.020239139986069313,1.020445340134926
AL,12698,0.014254213261931013,0.01923906599604634,1.0194253294201232
CT,10952,0.01524835646457268,0.01885100774188335,1.019029809753838
MN,21363,0.015400458737068764,0.01492401337539687,1.0150359325308889
TX,85561,0.014632835053353747,0.01279483260225002,1.0128770366946405
OH,43942,0.01458741067771153,0.00922262209059605,1.009265281512766


Lowest adjusted state effects with meaningful footprint


property_state,active_loan_count,active_delinquency_rate,coefficient,odds_ratio
OR,13655,0.008128890516294398,-0.08174505826185198,0.9215068593125192
WA,23885,0.00828972158258321,-0.07242017068455847,0.9301399961357529
VA,25698,0.009105766985757647,-0.056883303089655196,0.9447043070080194
CO,22841,0.009806926141587496,-0.03946194704082061,0.9613065338528994
UT,13354,0.011532125205930808,-0.029937881459148215,0.9705058180814026
MA,17410,0.011774842044801838,-0.02741770425705551,0.9729547492927774
KY,13310,0.011795642374154772,-0.025806286656258226,0.9745238495989565
CA,69537,0.011763521578440255,-0.019686056892685336,0.9805064482334195
NJ,27009,0.012477322374023474,-0.013422766232758737,0.9866669173780888
NC,33007,0.012058048292786379,-0.011011258092863012,0.989049143905582


In [0]:
# Creating the core portfolio findings table that summarizes the main headline metrics for business users
core_portfolio_findings_pd = pd.DataFrame([{"metric":"Active loans",
                                            "value":float(active_portfolio_summary.collect()[0]["active_loans"]),
                                            "business_meaning":"Loans still active in the portfolio as of latest observation"
                                        },
                                        {
                                            "metric":"Active delinquency rate",
                                            "value":float(active_portfolio_summary.collect()[0]["active_delinquency_rate"]),
                                            "business_meaning":"Share of active loans currently delinquent"
                                        },
                                        {
                                            "metric":"Active serious delinquency rate",
                                            "value":float(active_portfolio_summary.collect()[0]["active_serious_delinquency_rate"]),
                                            "business_meaning":"Share of active loans currently 90D+ equivalent"
                                        },
                                        {
                                            "metric":"Active total UPB",
                                            "value":float(active_exposure_summary.collect()[0]["active_total_upb"]),
                                            "business_meaning":"Total active unpaid principal balance at risk",
                                        },
                                        {
                                            "metric":"Delinquent UPB share",
                                            "value":float(active_exposure_summary.collect()[0]["delinquent_upb_share"]),
                                            "business_meaning":"Share of active UPB currently tied to delinquent loans"
                                        },
                                        {
                                            "metric":"Serious delinquent UPB share",
                                            "value":float(active_exposure_summary.collect()[0]["serious_delinquent_upb_share"]),
                                            "business_meaning":"Share of active UPB currently tied to serious delinquency"
                                        },
                                        {
                                            "metric":"Latest terminated share",
                                            "value":float(overall_portfolio_summary.collect()[0]["share_terminated_by_latest_observation"]),
                                            "business_meaning":"Share of loans already terminated by latest observation"
                                        },
                                        {
                                            "metric":"Latest prepaid share",
                                            "value":float(overall_portfolio_summary.collect()[0]["share_prepaid_by_latest_observation"]),
                                            "business_meaning":"Share of loans already prepaid by latest observation"
                                        }
])

core_portfolio_findings_pd

,metric,value,business_meaning
0,Active loans,9.532990e+05,Loans still active in the portfolio as of late...
1,Active delinquency rate,1.361168e-02,Share of active loans currently delinquent
2,Active serious delinquency rate,4.113085e-03,Share of active loans currently 90D+ equivalent
3,Active total UPB,3.086090e+11,Total active unpaid principal balance at risk
4,Delinquent UPB share,1.240990e-02,Share of active UPB currently tied to delinque...
5,Serious delinquent UPB share,3.940115e-03,Share of active UPB currently tied to serious ...
6,Latest terminated share,9.071668e-02,Share of loans already terminated by latest ob...
7,Latest prepaid share,8.759480e-02,Share of loans already prepaid by latest obser...


In [0]:
# Converting the credit, LTV and DTI risk ladders into pandas tables for final presentation and export
credit_risk_ladder_pd = credit_story_table.toPandas()
ltv_risk_ladder_pd = ltv_story_table.toPandas()
dti_risk_ladder_pd = dti_story_table.toPandas()

print("Credit risk ladder")
display(credit_risk_ladder_pd)

print("LTV risk ladder")
display(ltv_risk_ladder_pd)

print("DTI risk ladder")
display(dti_risk_ladder_pd)

Credit risk ladder


credit_score_band,active_loan_count,active_total_upb,delinquent_upb_share,serious_delinquent_upb_share
<620,1701,3.857100426700001E8,0.07982097338424997,0.02304394598717903
620-679,75836,1.9831334431989975E10,0.061033254294151185,0.01944306321908509
680-739,245810,7.518498158080011E10,0.022251461626443023,0.007021761730733961
740-799,508964,1.751737722673095E11,0.004790869396871554,0.0015360398660559866
800+,120668,3.7963278042289894E10,0.0019839258273771825,6.349903502312505E-4
Missing,320,6.992792372E7,0.016055182254451756,0.005341314887248307


LTV risk ladder


ltv_band,active_loan_count,active_total_upb,delinquent_upb_share,serious_delinquent_upb_share
<60,183162,4.2921243627860016E10,0.007917486718148556,0.001696818204557489
60-79,235918,7.810435928397014E10,0.009185523667399735,0.002528752856699198
80-89,233553,8.39723035407502E10,0.00946795656301341,0.0028210279212483734
90-100,300639,1.036029433652701E11,0.01908733531380442,0.00684083397342533


DTI risk ladder


dti_band,active_loan_count,active_total_upb,delinquent_upb_share,serious_delinquent_upb_share
<20,40387,1.0414394857030003E10,0.004445412296687478,0.0011099024128317341
20-29,147521,4.3194863316309944E10,0.004999028070508239,0.0013612617000178793
30-39,284982,9.037914032310014E10,0.008669396302276354,0.0025468602041036144
40-49,425394,1.4452128917817032E11,0.0162668075050295,0.005224251053761314
50+,55005,2.0096266933639973E10,0.021553598531523162,0.007981477816733381
Missing,10,3049680.5300000003,0.0,0.0


In [0]:
# Creating the final state findings table by ranking states by their share of portfolio delinquent exposure
state_findings_pd = (elevated_risk_state_watchlist.select("property_state", "active_loan_count", "active_total_upb",
                                                          "delinquent_upb_share", "serious_delinquent_upb_share",
                                                          "share_of_portfolio_delinquent_upb",
                                                          "share_of_portfolio_serious_delinquent_upb",
                                                          "delinquent_upb_risk_index",
                                                          "serious_delinquent_upb_risk_index"
                                                ).toPandas()
                                                .sort_values(["share_of_portfolio_delinquent_upb", 
                                                              "delinquent_upb_risk_index"],
                                                             ascending = [False, False]
                                                )
                                                .reset_index(drop = True)
)

state_findings_pd

,property_state,active_loan_count,active_total_upb,delinquent_upb_share,serious_delinquent_upb_share,share_of_portfolio_delinquent_upb,share_of_portfolio_serious_delinquent_upb,delinquent_upb_risk_index,serious_delinquent_upb_risk_index
0,FL,74394,2.486235e+10,0.017784,0.006631,0.115452,0.135574,1.433069,1.682844
1,TX,85561,2.802096e+10,0.013681,0.004108,0.100099,0.094661,1.102440,1.042554
2,NY,33478,1.214670e+10,0.016768,0.004648,0.053183,0.046428,1.351208,1.179587
3,AZ,25753,9.007929e+09,0.013316,0.004808,0.031320,0.035618,1.072998,1.220277
4,IN,26922,5.971185e+09,0.013978,0.003808,0.021793,0.018702,1.126328,0.966586


In [0]:
# Creating the final segment findings table by ranking borrower segments by their contribution to delinquent exposure
segment_findings_pd = (elevated_risk_segment_watchlist.select("credit_score_band", "ltv_band", "dti_band",
                                                              "active_loan_count", "active_total_upb",
                                                              "share_of_portfolio_delinquent_upb",
                                                              "share_of_portfolio_serious_delinquent_upb",
                                                              "delinquent_upb_risk_index",
                                                              "serious_delinquent_upb_risk_index"
                                                    ).toPandas()
                                                    .sort_values(["share_of_portfolio_delinquent_upb",
                                                                  "delinquent_upb_risk_index"],
                                                                 ascending = [False, False]
                                                    )
                                                    .reset_index(drop = True)
)

segment_findings_pd.head(20)

,credit_score_band,ltv_band,dti_band,active_loan_count,active_total_upb,share_of_portfolio_delinquent_upb,share_of_portfolio_serious_delinquent_upb,delinquent_upb_risk_index,serious_delinquent_upb_risk_index
0,680-739,90-100,40-49,55607,1.871296e+10,0.167957,0.179726,2.769907,2.963991
1,620-679,90-100,40-49,12792,3.916130e+09,0.084279,0.096180,6.641580,7.579396
2,680-739,80-89,40-49,27370,9.632479e+09,0.051051,0.051199,1.635603,1.640339
3,680-739,60-79,40-49,26953,8.538736e+09,0.044603,0.034677,1.612068,1.253317
4,680-739,90-100,30-39,27212,8.141351e+09,0.044238,0.046000,1.676894,1.743677
5,620-679,80-89,40-49,8419,2.696676e+09,0.043757,0.045748,5.007604,5.235397
6,620-679,60-79,40-49,10221,2.924419e+09,0.039752,0.035974,4.194938,3.796291
7,680-739,90-100,50+,7906,2.910134e+09,0.033717,0.041679,3.575607,4.419911
8,620-679,90-100,30-39,6479,1.602863e+09,0.031188,0.034714,6.004864,6.683641
9,620-679,<60,40-49,8274,1.733231e+09,0.020059,0.013710,3.571611,2.441109


In [0]:
# Creating the 30D watchlist table by keeping segments with enough observations and ranking those most likely to worsen to 60D
transition_30d_watchlist_pd = (transition_30d_segment_summary.filter(col("loan_months_30d") >= 300)
                                                             .select("credit_score_band", "ltv_band", "dti_band",
                                                                     "loan_months_30d", "cure_to_current_rate",
                                                                     "stay_30d_rate", "roll_to_60d_rate",
                                                                     "roll_to_90d_plus_rate"
                                                            ).toPandas()
                                                             .sort_values(["roll_to_60d_rate", 
                                                                           "cure_to_current_rate", 
                                                                           "loan_months_30d"],
                                                                          ascending = [False, True, False]
    )
    .reset_index(drop = True)
)

transition_30d_watchlist_pd.head(20)

,credit_score_band,ltv_band,dti_band,loan_months_30d,cure_to_current_rate,stay_30d_rate,roll_to_60d_rate,roll_to_90d_plus_rate
0,740-799,90-100,50+,729,0.460905,0.303155,0.234568,0.001372
1,620-679,90-100,20-29,568,0.348592,0.419014,0.228873,0.003521
2,620-679,60-79,50+,364,0.431319,0.340659,0.228022,0.000000
3,620-679,90-100,40-49,4607,0.314956,0.454092,0.227480,0.003473
4,680-739,90-100,50+,1453,0.375086,0.395045,0.226428,0.003441
5,620-679,90-100,50+,500,0.336000,0.436000,0.218000,0.010000
6,620-679,90-100,30-39,2190,0.346575,0.430594,0.217352,0.005479
7,680-739,90-100,40-49,8287,0.394111,0.386750,0.216484,0.002655
8,680-739,80-89,50+,536,0.466418,0.317164,0.214552,0.001866
9,620-679,80-89,40-49,2440,0.376230,0.415984,0.204508,0.003279


In [0]:
# Ranking segments where 60D delinquency is most often worsening into 90D+ instead of curing
distress_persistence_watchlist_pd = (distress_persistence_watchlist.toPandas()
                                                                    .sort_values(["roll_to_90d_plus_rate",
                                                                                 "cure_to_current_rate",
                                                                                 "loan_months_60d"],
                                                                                ascending = [False, True, False]
                                                                    )
                                                                    .reset_index(drop = True)
)

distress_persistence_watchlist_pd.head(20)

,credit_score_band,ltv_band,dti_band,loan_months_60d,cure_to_current_rate,improve_to_30d_rate,stay_60d_rate,roll_to_90d_plus_rate
0,740-799,90-100,50+,175,0.188571,0.085714,0.114286,0.611429
1,620-679,90-100,50+,133,0.097744,0.082707,0.210526,0.609023
2,680-739,80-89,50+,122,0.188525,0.065574,0.172131,0.573770
3,740-799,90-100,30-39,302,0.215232,0.056291,0.165563,0.562914
4,740-799,90-100,40-49,817,0.154223,0.100367,0.216646,0.528764
5,680-739,90-100,50+,418,0.093301,0.102871,0.296651,0.507177
6,740-799,<60,40-49,186,0.295699,0.091398,0.112903,0.500000
7,680-739,90-100,40-49,2177,0.127699,0.105650,0.271015,0.495636
8,620-679,60-79,50+,107,0.102804,0.102804,0.299065,0.495327
9,740-799,80-89,40-49,304,0.266447,0.088816,0.151316,0.493421


In [0]:
# Ranking borrower groups that are prepaying fastest within the first 6 months
early_prepay_findings_pd = (early_prepay_segment_summary.toPandas()
                                                        .sort_values(["prepay_by_month_6_share", "loan_count"],
                                                                     ascending = [False, False]
                                                        )
                                                        .reset_index(drop = True)
)

early_prepay_findings_pd.head(20)

,credit_score_band,loan_purpose,loan_count,prepay_by_month_6_share,terminated_by_month_6_share
0,800+,P,116639,0.063855,0.064687
1,800+,C,10268,0.063109,0.063888
2,740-799,C,50435,0.039179,0.040230
3,740-799,P,456079,0.031563,0.032619
4,680-739,C,45882,0.028813,0.030099
5,800+,N,11384,0.027934,0.028637
6,620-679,C,23041,0.027777,0.029122
7,740-799,N,52152,0.026001,0.027056
8,680-739,N,20246,0.023264,0.024795
9,620-679,N,6379,0.022261,0.024298


In [0]:
# Converting event-timing summaries to pandas and ranking the fastest delinquency and prepayment patterns

time_to_delinquency_pd = (time_to_delinquency_summary.toPandas()
                                                        .sort_values(["median_months_to_first_delinquency",
                                                                      "loan_count_with_delinquency"],
                                                                     ascending = [True, False]
                                                        )
                                                        .reset_index(drop = True)
)

time_to_prepay_pd = (time_to_prepay_summary.toPandas()
                                            .sort_values(["median_months_to_first_prepay", "loan_count_with_prepay"],
                                                         ascending = [True, False]
                                            )
                                            .reset_index(drop = True)
)

print("Shortest median time to first delinquency among loans that ever became delinquent "
      "(conditional event-time view, not unconditional portfolio risk)"
)
display(time_to_delinquency_pd.head(20))

print("Shortest median time to first prepayment among loans that ever prepaid "
      "(conditional event-time view, not unconditional portfolio risk)"
)
display(time_to_prepay_pd.head(20))

Shortest median time to first delinquency among loans that ever became delinquent (conditional event-time view, not unconditional portfolio risk)


credit_score_band,ltv_band,loan_count_with_delinquency,median_months_to_first_delinquency,p75_months_to_first_delinquency
800+,60-79,588,2,7
800+,80-89,512,2,7
740-799,60-79,3555,3,7
740-799,<60,2411,3,8
800+,<60,608,3,6
800+,90-100,451,3,8
740-799,80-89,3100,4,8
740-799,90-100,4615,5,9
680-739,60-79,3566,5,9
680-739,80-89,3098,5,9


Shortest median time to first prepayment among loans that ever prepaid (conditional event-time view, not unconditional portfolio risk)


credit_score_band,loan_purpose,loan_count_with_prepay,median_months_to_first_prepay,p75_months_to_first_prepay
800+,P,14912,6,9
740-799,P,37933,7,11
740-799,N,4474,7,10
800+,C,1636,7,10
800+,N,881,7,10
740-799,C,6236,8,11
680-739,C,5332,8,12
680-739,N,1708,8,11
620-679,N,624,8,11
680-739,P,11834,9,12


In [0]:
# Converting vintage findings to a presentation-ready table sorted by origination quarter
vintage_findings_pd = (vintage_presentation_summary.toPandas()
                                                    .sort_values("origination_quarter")
                                                    .reset_index(drop = True)
)

vintage_findings_pd

,origination_quarter,month_3_coverage_ratio,month_6_coverage_ratio,month_9_coverage_ratio,month_9_comparable,month_3_delinquency_rate,month_6_delinquency_rate,month_9_delinquency_rate,prepay_by_month_6_share,prepay_by_month_9_share,avg_credit_score_delta,avg_ltv_delta,avg_dti_delta,share_credit_below_680_delta,share_ltv_90_plus_delta,share_dti_45_plus_delta,share_first_time_homebuyer_delta
0,2024Q1,0.999563,0.999735,0.999479,Yes,0.005667,0.008080,0.011219,0.033666,0.057931,-1.080175,0.463844,0.100623,-0.000132,0.013691,0.006221,0.015284
1,2024Q2,0.999464,0.999460,0.999045,Yes,0.005578,0.009703,0.010533,0.040159,0.062103,-1.129083,0.273332,0.192567,0.006017,0.012222,0.011014,0.030827
2,2024Q3,0.999216,0.999432,0.999253,Yes,0.006079,0.007075,0.008699,0.026294,0.045501,-0.154318,0.138412,-0.142141,0.002881,-0.000903,-0.008626,-0.001339
3,2024Q4,0.999116,0.999620,0.712836,No,0.004657,0.005244,0.007822,0.025693,0.060880,2.010902,-0.747413,-0.102163,-0.008384,-0.020589,-0.005678,-0.038226


In [0]:
# Displaying the strongest positive and negative model associations for delinquency and prepayment

print("Active delinquency : largest positive baseline-relative associations")
display(delinquency_driver_table.head(15))

print("Active delinquency : largest negative / protective baseline-relative associations")
display(delinquency_driver_table.sort_values("odds_ratio", ascending = True).head(15))

print("Serious delinquency : largest positive baseline-relative associations")
display(serious_delinquency_driver_table.head(15))

print("Serious delinquency : largest negative / protective baseline-relative associations")
display(serious_delinquency_driver_table.sort_values("odds_ratio", ascending = True).head(15))

print("Month-6 prepayment : largest positive baseline-relative associations")
display(prepay_driver_table.head(15))

print("Month-6 prepayment : largest negative baseline-relative associations")
display(prepay_driver_table.sort_values("odds_ratio", ascending = True).head(15))

Active delinquency : largest positive baseline-relative associations


model_name,feature,coefficient,odds_ratio
active_delinquency,credit_score_band_ohe_<620,1.006190854508623,2.735162514794001
active_delinquency,credit_score_band_ohe_620-679,0.6879532034984689,1.9896389766286862
active_delinquency,property_type_ohe_MH,0.18189011943977865,1.1994823868418436
active_delinquency,credit_score_band_ohe_680-739,0.1493393889647824,1.1610669756668903
active_delinquency,dti_band_ohe_50+,0.14291359974664475,1.1536301234250177
active_delinquency,ltv_band_ohe_90-100,0.13838355456171947,1.1484159459857217
active_delinquency,dti_band_ohe_40-49,0.08677020160698615,1.090646022215783
active_delinquency,loan_purpose_ohe_C,0.08171426590841868,1.0851457024060172
active_delinquency,occupancy_status_ohe_P,0.06538902026289736,1.0675742516290812
active_delinquency,property_type_ohe_SF,0.05494737031141765,1.056485010735191


Active delinquency : largest negative / protective baseline-relative associations


model_name,feature,coefficient,odds_ratio
active_delinquency,credit_score_band_ohe_740-799,-0.23912306195173566,0.7873179875225944
active_delinquency,credit_score_band_ohe_800+,-0.19390561448878788,0.8237356493098802
active_delinquency,dti_band_ohe_<20,-0.11044901087352516,0.8954319863202931
active_delinquency,dti_band_ohe_20-29,-0.10129332257958831,0.9036679277994376
active_delinquency,occupancy_status_ohe_S,-0.07821252952724601,0.9247678651795334
active_delinquency,property_type_ohe_CP,-0.07603862764911903,0.9267804065195976
active_delinquency,ltv_band_ohe_<60,-0.07451271338547526,0.9281956734745774
active_delinquency,property_type_ohe_PU,-0.06075451851493885,0.9410542228119594
active_delinquency,occupancy_status_ohe_I,-0.0590613594066043,0.9426489270037407
active_delinquency,dti_band_ohe_30-39,-0.054614597497809995,0.9468499959999498


Serious delinquency : largest positive baseline-relative associations


...

model_name,feature,coefficient,odds_ratio
active_serious_delinquency,credit_score_band_ohe_740-799,-0.09739542477653568,0.9071972069460732
active_serious_delinquency,credit_score_band_ohe_800+,-0.07197245697277453,0.9305565258019654
active_serious_delinquency,dti_band_ohe_<20,-0.04912931110379392,0.9520580100661511
active_serious_delinquency,occupancy_status_ohe_S,-0.0480929600136262,0.9530451878658261
active_serious_delinquency,dti_band_ohe_20-29,-0.04709114754671227,0.9540004388278308
active_serious_delinquency,property_type_ohe_CP,-0.04683131924007874,0.9542483473518104
active_serious_delinquency,occupancy_status_ohe_I,-0.044105851642108215,0.956852667686166
active_serious_delinquency,ltv_band_ohe_<60,-0.043821067132270054,0.9571252033092402
active_serious_delinquency,ltv_band_ohe_60-79,-0.02697475692496766,0.9733858124653119
active_serious_delinquency,first_time_homebuyer_flag_ohe_N,-0.026868952422347533,0.9734888065155783


Month-6 prepayment : largest positive baseline-relative associations


model_name,feature,coefficient,odds_ratio
month_6_prepay,credit_score_band_ohe_800+,0.3425879942228641,1.408588295841479
month_6_prepay,first_time_homebuyer_flag_ohe_N,0.2558131391499357,1.291511372477534
month_6_prepay,property_type_ohe_PU,0.13032276243440427,1.1391960136714336
month_6_prepay,ltv_band_ohe_<60,0.0924688825856533,1.096879008990637
month_6_prepay,dti_band_ohe_50+,0.05540050831196101,1.0569638527231167
month_6_prepay,loan_purpose_ohe_P,0.054424612240852016,1.0559328690002456
month_6_prepay,occupancy_status_ohe_S,0.052811640407047195,1.0542310518846614
month_6_prepay,dti_band_ohe_<20,0.04581504248110598,1.0468807645805178
month_6_prepay,ltv_band_ohe_60-79,0.04487911655965147,1.0459014201047039
month_6_prepay,occupancy_status_ohe_P,0.03882358440361008,1.0395870680993335


Month-6 prepayment : largest negative baseline-relative associations


model_name,feature,coefficient,odds_ratio
month_6_prepay,first_time_homebuyer_flag_ohe_Y,-0.2558131391499329,0.7742866391348001
month_6_prepay,property_type_ohe_CP,-0.17669474044899275,0.8380355638414141
month_6_prepay,credit_score_band_ohe_620-679,-0.17241593552004145,0.8416290369453379
month_6_prepay,credit_score_band_ohe_680-739,-0.16848019963037544,0.8449479935358901
month_6_prepay,credit_score_band_ohe_<620,-0.16353641724158285,0.8491355752631278
month_6_prepay,property_type_ohe_MH,-0.1349008199639354,0.8738025711610397
month_6_prepay,loan_purpose_ohe_N,-0.1320061508495806,0.8763356048543085
month_6_prepay,ltv_band_ohe_90-100,-0.11549864169484707,0.8909217823799042
month_6_prepay,property_type_ohe_SF,-0.1126498923097755,0.8934634137761873
month_6_prepay,occupancy_status_ohe_I,-0.06520776779306987,0.9368727911625885


In [0]:
# Validating the final headline metrics against the supporting summary tables
# - Pulling the top state and top segment so the final findings can be checked against source tables
# - Using lookup tables so key benchmark values can be retrieved directly by band

top_state_row = state_findings_pd.iloc[0]
top_segment_row = segment_findings_pd.iloc[0]

credit_lookup = credit_risk_ladder_pd.set_index("credit_score_band")
ltv_lookup = ltv_risk_ladder_pd.set_index("ltv_band")
dti_lookup = dti_risk_ladder_pd.set_index("dti_band")

final_finding_validation_table = pd.DataFrame([{"metric_name" : "active_loans",
                                                "metric_value" : float(core_portfolio_findings_pd.loc[core_portfolio_findings_pd["metric"] == "Active loans", "value"].iloc[0])
                                            },
                                            {
                                                "metric_name" : "active_total_upb",
                                                "metric_value" : float(core_portfolio_findings_pd.loc[core_portfolio_findings_pd["metric"] == "Active total UPB", "value"].iloc[0])
                                            },
                                            {
                                                "metric_name" : "delinquent_upb_share",
                                                "metric_value" : float(core_portfolio_findings_pd.loc[core_portfolio_findings_pd["metric"] == "Delinquent UPB share", "value"].iloc[0])
                                            },
                                            {
                                                "metric_name" : "serious_delinquent_upb_share",
                                                "metric_value" : float(core_portfolio_findings_pd.loc[core_portfolio_findings_pd["metric"] == "Serious delinquent UPB share", "value"].iloc[0])
                                            },
                                            {
                                                "metric_name" : "top_state_share_of_portfolio_delinquent_upb","metric_value" : float(top_state_row["share_of_portfolio_delinquent_upb"])
                                            },
                                            {
                                                "metric_name" : "top_segment_share_of_portfolio_delinquent_upb","metric_value" : float(top_segment_row["share_of_portfolio_delinquent_upb"])
                                            },
                                            {
                                                "metric_name" : "credit_<620_delinquent_upb_share",
                                                "metric_value" : float(credit_lookup.loc["<620", "delinquent_upb_share"])
                                            },
                                            {
                                                "metric_name" : "credit_800+_delinquent_upb_share",
                                                "metric_value" : float(credit_lookup.loc["800+", "delinquent_upb_share"])
                                            },
                                            {
                                                "metric_name" : "ltv_90_100_delinquent_upb_share",
                                                "metric_value" : float(ltv_lookup.loc["90-100", "delinquent_upb_share"])
                                            },
                                            {
                                                "metric_name" : "dti_50_plus_delinquent_upb_share",
                                                "metric_value" : float(dti_lookup.loc["50+", "delinquent_upb_share"])
                                            }
])

final_finding_validation_table

,metric_name,metric_value
0,active_loans,9.532990e+05
1,active_total_upb,3.086090e+11
2,delinquent_upb_share,1.240990e-02
3,serious_delinquent_upb_share,3.940115e-03
4,top_state_share_of_portfolio_delinquent_upb,1.154518e-01
5,top_segment_share_of_portfolio_delinquent_upb,1.679574e-01
6,credit_<620_delinquent_upb_share,7.982097e-02
7,credit_800+_delinquent_upb_share,1.983926e-03
8,ltv_90_100_delinquent_upb_share,1.908734e-02
9,dti_50_plus_delinquent_upb_share,2.155360e-02


In [0]:
# Building the final business findings table by turning analysis results into stakeholder-ready insights
def pct(value) : 
    return f"{float(value):.2%}"

def billions(value) : 
    return f"${float(value) / 1_000_000_000:.2f}B"

top_state_row = state_findings_pd.iloc[0]
top_segment_row = segment_findings_pd.iloc[0]

credit_lookup = credit_risk_ladder_pd.set_index("credit_score_band")
ltv_lookup = ltv_risk_ladder_pd.set_index("ltv_band")
dti_lookup = dti_risk_ladder_pd.set_index("dti_band")

top_distress_rows = distress_persistence_watchlist_pd.head(3).copy()
distress_examples_text = "; ".join(
    [
        f"{row.credit_score_band} / {row.ltv_band} / {row.dti_band} rolls from 60D to 90D+ at {pct(row.roll_to_90d_plus_rate)}"
        for row in top_distress_rows.itertuples(index = False)
    ]
)

top_adjusted_states = state_adjusted_state_effects_filtered.head(4).copy()
adjusted_state_text = ", ".join(
    [
        f"{row.property_state} ({row.odds_ratio:.3f})"
        for row in top_adjusted_states.itertuples(index = False)
    ]
)

highest_month6_vintage = (vintage_findings_pd.dropna(subset = ["month_6_delinquency_rate"])
                                             .sort_values(["month_6_delinquency_rate", "origination_quarter"], 
                                                          ascending = [False, True])
                                             .iloc[0]
)

lowest_month6_vintage = (vintage_findings_pd.dropna(subset = ["month_6_delinquency_rate"])
                                            .sort_values(["month_6_delinquency_rate", "origination_quarter"], 
                                                         ascending = [True, True])
                                            .iloc[0]
)

prepay_800_purchase = time_to_prepay_pd[(time_to_prepay_pd["credit_score_band"] == "800+") &
                                        (time_to_prepay_pd["loan_purpose"] == "P")
]

prepay_620_purchase = time_to_prepay_pd[(time_to_prepay_pd["credit_score_band"] == "620-679") &
                                        (time_to_prepay_pd["loan_purpose"] == "P")
]

prepay_800_purchase_median = (int(prepay_800_purchase["median_months_to_first_prepay"].iloc[0])
                              if not prepay_800_purchase.empty else None
)

prepay_620_purchase_median = (int(prepay_620_purchase["median_months_to_first_prepay"].iloc[0])
                              if not prepay_620_purchase.empty else None
)

final_business_findings_table = pd.DataFrame([
    {
        "finding_id":1,
        "finding_title":"Portfolio risk is concentrated rather than broad-based",
        "business_finding":"A relatively small set of states and borrower segments contributes a disproportionate share of delinquent and serious-delinquent UPB.",
        "primary_evidence_table":"state_findings_pd / segment_findings_pd",
        "key_numbers":(
            f"{top_state_row['property_state']} contributes {pct(top_state_row['share_of_portfolio_delinquent_upb'])} "
            f"of portfolio delinquent UPB and {pct(top_state_row['share_of_portfolio_serious_delinquent_upb'])} of serious delinquent UPB; "
            f"the top segment {top_segment_row['credit_score_band']} / {top_segment_row['ltv_band']} / {top_segment_row['dti_band']} "
            f"contributes {pct(top_segment_row['share_of_portfolio_delinquent_upb'])} of portfolio delinquent UPB and "
            f"{pct(top_segment_row['share_of_portfolio_serious_delinquent_upb'])} of serious delinquent UPB."
        ),
        "why_it_matters":"Risk management should focus on concentrated pockets of exposure rather than treating the portfolio as uniformly risky."
    },
    {
        "finding_id":2,
        "finding_title":"Credit score is the strongest descriptive risk separator",
        "business_finding":"Observed delinquent and serious-delinquent exposure falls sharply as borrower credit quality improves.",
        "primary_evidence_table":"credit_risk_ladder_pd / delinquency_driver_table / serious_delinquency_driver_table",
        "key_numbers":(
            f"<620 delinquent UPB share = {pct(credit_lookup.loc['<620', 'delinquent_upb_share'])}; "
            f"620-679 = {pct(credit_lookup.loc['620-679', 'delinquent_upb_share'])}; "
            f"680-739 = {pct(credit_lookup.loc['680-739', 'delinquent_upb_share'])}; "
            f"740-799 = {pct(credit_lookup.loc['740-799', 'delinquent_upb_share'])}; "
            f"800+ = {pct(credit_lookup.loc['800+', 'delinquent_upb_share'])}. "
            f"Baseline-relative model odds ratio for active delinquency : <620 = "
            f"{delinquency_driver_table.loc[delinquency_driver_table['feature'] == 'credit_score_band_ohe_<620', 'odds_ratio'].iloc[0]:.2f}, "
            f"620-679 = {delinquency_driver_table.loc[delinquency_driver_table['feature'] == 'credit_score_band_ohe_620-679', 'odds_ratio'].iloc[0]:.2f}."
        ),
        "why_it_matters":"Credit quality should remain a primary dimension for underwriting, surveillance, pricing and capital allocation."
    },
    {
        "finding_id":3,
        "finding_title":"High LTV materially amplifies delinquency stress",
        "business_finding":"Leverage is a clear stress amplifier, especially in the 90-100 LTV band.",
        "primary_evidence_table":"ltv_risk_ladder_pd / ltv_roll_rate_summary / delinquency_driver_table",
        "key_numbers":(
            f"90-100 LTV delinquent UPB share = {pct(ltv_lookup.loc['90-100', 'delinquent_upb_share'])} "
            f"versus {pct(ltv_lookup.loc['<60', 'delinquent_upb_share'])} for <60 LTV. "
            f"Baseline-relative model odds ratio for active delinquency, 90-100 LTV = "
            f"{delinquency_driver_table.loc[delinquency_driver_table['feature'] == 'ltv_band_ohe_90-100', 'odds_ratio'].iloc[0]:.2f}."
        ),
        "why_it_matters":"High leverage reduces borrower resilience and should be closely monitored in portfolio surveillance and stress testing."
    },
    {
        "finding_id":4,
        "finding_title":"High DTI adds clear payment-stress risk",
        "business_finding":"Observed delinquent exposure rises meaningfully with debt burden, especially above 40%.",
        "primary_evidence_table":"dti_risk_ladder_pd / dti_roll_rate_summary / delinquency_driver_table",
        "key_numbers":(
            f"Delinquent UPB share rises from {pct(dti_lookup.loc['<20', 'delinquent_upb_share'])} for <20 DTI "
            f"to {pct(dti_lookup.loc['50+', 'delinquent_upb_share'])} for 50+ DTI. "
            f"Baseline-relative model odds ratio for active delinquency, 50+ DTI = "
            f"{delinquency_driver_table.loc[delinquency_driver_table['feature'] == 'dti_band_ohe_50+', 'odds_ratio'].iloc[0]:.2f}."
        ),
        "why_it_matters":"DTI is a strong payment-stress signal and should be used alongside credit score and LTV in watchlist design."
    },
    {
        "finding_id":5,
        "finding_title":"Risk is interaction-based, not single-factor",
        "business_finding":"The most important risk pockets are combinations of moderate-to-weak credit, high LTV and elevated DTI.",
        "primary_evidence_table":"segment_findings_pd / top_segment_risk_insight / credit_ltv_risk_summary",
        "key_numbers":(
            f"The top flagged segment is {top_segment_row['credit_score_band']} / {top_segment_row['ltv_band']} / {top_segment_row['dti_band']}; "
            f"it contributes {pct(top_segment_row['share_of_portfolio_delinquent_upb'])} of portfolio delinquent UPB and "
            f"{pct(top_segment_row['share_of_portfolio_serious_delinquent_upb'])} of serious delinquent UPB, with "
            f"a delinquent UPB risk index of {top_segment_row['delinquent_upb_risk_index']:.2f} and "
            f"a serious delinquent UPB risk index of {top_segment_row['serious_delinquent_upb_risk_index']:.2f}."
        ),
        "why_it_matters":"Portfolio risk policy should target interacting borrower characteristics rather than relying on one-dimensional thresholds."
    },
    {
        "finding_id":6,
        "finding_title":"Distress persistence is strong in selected active-loan segments",
        "business_finding":"Among loans that remain active into the next monthly observation, several 60D-delinquent segments have high probabilities of rolling to 90D+ rather than curing.",
        "primary_evidence_table":"distress_persistence_watchlist_pd / transition_30d_watchlist_pd",
        "key_numbers":distress_examples_text,
        "why_it_matters":"These segments are candidates for early intervention and elevated servicing attention, but the transition view should be read as active-to-active behavior rather than full-loan exit behavior."
    },
    {
        "finding_id":7,
        "finding_title":"Prepayment follows a different pattern from credit distress",
        "business_finding":"Higher-credit borrowers prepay earlier and at higher rates, especially in purchase and cash-out segments.",
        "primary_evidence_table":"early_prepay_findings_pd / time_to_prepay_pd / prepay_driver_table",
        "key_numbers":(
            f"800+ / Purchase month-6 prepay share = "
            f"{pct(early_prepay_findings_pd.loc[(early_prepay_findings_pd['credit_score_band'] == '800+') & (early_prepay_findings_pd['loan_purpose'] == 'P'), 'prepay_by_month_6_share'].iloc[0])}; "
            f"baseline-relative model odds ratio for 800+ credit in month-6 prepayment = "
            f"{prepay_driver_table.loc[prepay_driver_table['feature'] == 'credit_score_band_ohe_800+', 'odds_ratio'].iloc[0]:.2f}; "
            f"conditional median time to first prepay is {prepay_800_purchase_median} months for 800+ / Purchase "
            f"versus {prepay_620_purchase_median} months for 620-679 / Purchase."
        ),
        "why_it_matters":"Safer borrowers may still create portfolio runoff and reinvestment risk, which matters to investors and servicing strategy."
    },
    {
        "finding_id":8,
        "finding_title":"Vintage differences are partly compositional",
        "business_finding":"Observed vintage differences should be interpreted alongside borrower-mix changes and coverage ratios rather than as pure seasoning effects.",
        "primary_evidence_table":"vintage_findings_pd / vintage_composition_delta_summary / month_9_vintage_comparable_summary",
        "key_numbers":(
            f"{highest_month6_vintage['origination_quarter']} has the highest month-6 delinquency rate at {pct(highest_month6_vintage['month_6_delinquency_rate'])}, "
            f"while {lowest_month6_vintage['origination_quarter']} has the lowest at {pct(lowest_month6_vintage['month_6_delinquency_rate'])}. "
            f"For {highest_month6_vintage['origination_quarter']}, borrower-mix deltas include below-680 share "
            f"{highest_month6_vintage['share_credit_below_680_delta']:+.2%}, 90+ LTV share {highest_month6_vintage['share_ltv_90_plus_delta']:+.2%}, "
            f"45+ DTI share {highest_month6_vintage['share_dti_45_plus_delta']:+.2%} and first-time-homebuyer share "
            f"{highest_month6_vintage['share_first_time_homebuyer_delta']:+.2%}. "
            f"Month-9 comparisons should exclude vintages where comparability is flagged as 'No'."
        ),
        "why_it_matters":"Vintage comparisons should be interpreted through borrower composition and coverage, not seasoning alone."
    },
    {
        "finding_id":9,
        "finding_title":"Some state risk remains after borrower-mix adjustment",
        "business_finding":"A few states remain elevated even after controlling for observed borrower and loan characteristics.",
        "primary_evidence_table":"state_adjusted_state_effects_filtered / state_risk_comparison",
        "key_numbers":f"Highest positive baseline-relative adjusted state associations with meaningful footprint : {adjusted_state_text}.",
        "why_it_matters":"Geography may capture residual local risk conditions not fully explained by observable borrower mix."
    },
    {
        "finding_id":10,
        "finding_title":"The portfolio has meaningful active exposure at risk",
        "business_finding":"The portfolio still carries large active UPB with non-trivial delinquent and serious-delinquent exposure.",
        "primary_evidence_table":"core_portfolio_findings_pd",
        "key_numbers":(
            f"Active loans = {int(core_portfolio_findings_pd.loc[core_portfolio_findings_pd['metric'] == 'Active loans', 'value'].iloc[0]) : ,}; "
            f"active total UPB = {billions(core_portfolio_findings_pd.loc[core_portfolio_findings_pd['metric'] == 'Active total UPB', 'value'].iloc[0])}; "
            f"delinquent UPB share = {pct(core_portfolio_findings_pd.loc[core_portfolio_findings_pd['metric'] == 'Delinquent UPB share', 'value'].iloc[0])}; "
            f"serious delinquent UPB share = {pct(core_portfolio_findings_pd.loc[core_portfolio_findings_pd['metric'] == 'Serious delinquent UPB share', 'value'].iloc[0])}."
        ),
        "why_it_matters":"Even modest delinquency rates matter materially when applied to a very large unpaid principal base."
    }
])

final_business_findings_table

,finding_id,finding_title,business_finding,primary_evidence_table,key_numbers,why_it_matters
0,1,Portfolio risk is concentrated rather than bro...,A relatively small set of states and borrower ...,state_findings_pd / segment_findings_pd,FL contributes 11.55% of portfolio delinquent ...,Risk management should focus on concentrated p...
1,2,Credit score is the strongest descriptive risk...,Observed delinquent and serious-delinquent exp...,credit_risk_ladder_pd / delinquency_driver_tab...,<620 delinquent UPB share = 7.98%; 620-679 = 6...,Credit quality should remain a primary dimensi...
2,3,High LTV materially amplifies delinquency stress,"Leverage is a clear stress amplifier, especial...",ltv_risk_ladder_pd / ltv_roll_rate_summary / d...,90-100 LTV delinquent UPB share = 1.91% versus...,High leverage reduces borrower resilience and ...
3,4,High DTI adds clear payment-stress risk,Observed delinquent exposure rises meaningfull...,dti_risk_ladder_pd / dti_roll_rate_summary / d...,Delinquent UPB share rises from 0.44% for <20 ...,DTI is a strong payment-stress signal and shou...
4,5,"Risk is interaction-based, not single-factor",The most important risk pockets are combinatio...,segment_findings_pd / top_segment_risk_insight...,The top flagged segment is 680-739 / 90-100 / ...,Portfolio risk policy should target interactin...
5,6,Distress persistence is strong in selected act...,Among loans that remain active into the next m...,distress_persistence_watchlist_pd / transition...,740-799 / 90-100 / 50+ rolls from 60D to 90D+ ...,These segments are candidates for early interv...
6,7,Prepayment follows a different pattern from cr...,Higher-credit borrowers prepay earlier and at ...,early_prepay_findings_pd / time_to_prepay_pd /...,800+ / Purchase month-6 prepay share = 6.39%; ...,Safer borrowers may still create portfolio run...
7,8,Vintage differences are partly compositional,Observed vintage differences should be interpr...,vintage_findings_pd / vintage_composition_delt...,2024Q2 has the highest month-6 delinquency rat...,Vintage comparisons should be interpreted thro...
8,9,Some state risk remains after borrower-mix adj...,A few states remain elevated even after contro...,state_adjusted_state_effects_filtered / state_...,Highest positive baseline-relative adjusted st...,Geography may capture residual local risk cond...
9,10,The portfolio has meaningful active exposure a...,The portfolio still carries large active UPB w...,core_portfolio_findings_pd,"Active loans = 953,299; active total UPB = $3...",Even modest delinquency rates matter materiall...


In [0]:
# Keeping only the headline portfolio metrics that are most useful for dashboards

final_business_findings_table_for_tableau = final_business_findings_table.copy()

numeric_extracts = {
    "active_loans" : float(active_portfolio_summary.collect()[0]["active_loans"]),
    "active_delinquency_rate" : float(active_portfolio_summary.collect()[0]["active_delinquency_rate"]),
    "active_serious_delinquency_rate" : float(active_portfolio_summary.collect()[0]["active_serious_delinquency_rate"]),
    "active_total_upb" : float(active_exposure_summary.collect()[0]["active_total_upb"]),
    "delinquent_upb_share" : float(active_exposure_summary.collect()[0]["delinquent_upb_share"]),
    "serious_delinquent_upb_share" : float(active_exposure_summary.collect()[0]["serious_delinquent_upb_share"]),
}

pd.DataFrame([numeric_extracts])

,active_loans,active_delinquency_rate,active_serious_delinquency_rate,active_total_upb,delinquent_upb_share,serious_delinquent_upb_share
0,953299.0,0.013612,0.004113,3.086090e+11,0.01241,0.00394


In [0]:
# Grouping all final output tables in one dictionary so they can be saved consistently
final_export_tables = {"core_portfolio_findings" : core_portfolio_findings_pd,
                       "credit_risk_ladder" : credit_risk_ladder_pd,
                       "ltv_risk_ladder" : ltv_risk_ladder_pd,
                       "dti_risk_ladder" : dti_risk_ladder_pd,
                       "state_findings" : state_findings_pd,
                       "segment_findings" : segment_findings_pd,
                       "transition_30d_watchlist" : transition_30d_watchlist_pd,
                       "distress_persistence_watchlist" : distress_persistence_watchlist_pd,
                       "early_prepay_findings" : early_prepay_findings_pd,
                       "time_to_delinquency" : time_to_delinquency_pd,
                       "time_to_prepay" : time_to_prepay_pd,
                       "vintage_findings" : vintage_findings_pd,
                       "delinquency_drivers" : delinquency_driver_table,
                       "serious_delinquency_drivers" : serious_delinquency_driver_table,
                       "prepay_drivers" : prepay_driver_table,
                       "state_adjusted_effects" : state_adjusted_state_effects_filtered,
                       "model_quality_summary" : model_quality_summary,
                       "final_business_findings" : final_business_findings_table
}

list(final_export_tables.keys())

['core_portfolio_findings',
 'credit_risk_ladder',
 'ltv_risk_ladder',
 'dti_risk_ladder',
 'state_findings',
 'segment_findings',
 'transition_30d_watchlist',
 'distress_persistence_watchlist',
 'early_prepay_findings',
 'time_to_delinquency',
 'time_to_prepay',
 'vintage_findings',
 'delinquency_drivers',
 'serious_delinquency_drivers',
 'prepay_drivers',
 'state_adjusted_effects',
 'model_quality_summary',
 'final_business_findings']

In [0]:
# Saving each final analysis output as a managed Databricks Delta table
for table_name, table_df in final_export_tables.items() : 
    spark_df = spark.createDataFrame(table_df)
    full_table_name = f"{OUTPUT_TABLE_PREFIX}.{table_name}"

    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(full_table_name)
    )

    print(f"Saved : {full_table_name}")

Saved : mortgage_risk.analytics.core_portfolio_findings
Saved : mortgage_risk.analytics.credit_risk_ladder
Saved : mortgage_risk.analytics.ltv_risk_ladder
Saved : mortgage_risk.analytics.dti_risk_ladder
Saved : mortgage_risk.analytics.state_findings
Saved : mortgage_risk.analytics.segment_findings
Saved : mortgage_risk.analytics.transition_30d_watchlist
Saved : mortgage_risk.analytics.distress_persistence_watchlist
Saved : mortgage_risk.analytics.early_prepay_findings
Saved : mortgage_risk.analytics.time_to_delinquency
Saved : mortgage_risk.analytics.time_to_prepay
Saved : mortgage_risk.analytics.vintage_findings
Saved : mortgage_risk.analytics.delinquency_drivers
Saved : mortgage_risk.analytics.serious_delinquency_drivers
Saved : mortgage_risk.analytics.prepay_drivers
Saved : mortgage_risk.analytics.state_adjusted_effects
Saved : mortgage_risk.analytics.model_quality_summary
Saved : mortgage_risk.analytics.final_business_findings


In [0]:
# Verifying that key final tables are being saved correctly and are readable from Databricks
display(spark.table("mortgage_risk.analytics.core_portfolio_findings"))
display(spark.table("mortgage_risk.analytics.state_findings"))
display(spark.table("mortgage_risk.analytics.final_business_findings"))

metric,value,business_meaning
Active loans,953299.0,Loans still active in the portfolio as of latest observation
Active delinquency rate,0.013611679022006737,Share of active loans currently delinquent
Active serious delinquency rate,0.004113085191529625,Share of active loans currently 90D+ equivalent
Active total UPB,3.086090042887791E11,Total active unpaid principal balance at risk
Delinquent UPB share,0.01240990381287216,Share of active UPB currently tied to delinquent loans
Serious delinquent UPB share,0.003940114837648019,Share of active UPB currently tied to serious delinquency
Latest terminated share,0.0907166777787634,Share of loans already terminated by latest observation
Latest prepaid share,0.08759479858490071,Share of loans already prepaid by latest observation


property_state,active_loan_count,active_total_upb,delinquent_upb_share,serious_delinquent_upb_share,share_of_portfolio_delinquent_upb,share_of_portfolio_serious_delinquent_upb,delinquent_upb_risk_index,serious_delinquent_upb_risk_index
FL,74394,2.486234734139997E10,0.01778424962810058,0.006630599036217848,0.11545178887484432,0.1355743161512687,1.433069095157199,1.6828441071976128
TX,85561,2.802095745280001E10,0.01368117067326308,0.004107781143234924,0.10009888104917139,0.0946613719364502,1.1024397029630721,1.0425536596001832
NY,33478,1.2146699665940008E10,0.016768359748049935,0.004647706419242491,0.05318288191253403,0.0464279499417434,1.3512078740414548,1.179586537639308
AZ,25753,9.007928863810007E9,0.01331580122506282,0.00480803185002966,0.03131953047041379,0.035618433118318596,1.0729979398592129,1.2202770853500626
IN,26922,5.97118464756E9,0.013977619466868334,0.003808458443718139,0.02179298426552871,0.018702180710191064,1.1263277844563198,0.9665856455065989


finding_id,finding_title,business_finding,primary_evidence_table,key_numbers,why_it_matters
1,Portfolio risk is concentrated rather than broad-based,A relatively small set of states and borrower segments contributes a disproportionate share of delinquent and serious-delinquent UPB.,state_findings_pd / segment_findings_pd,FL contributes 11.55% of portfolio delinquent UPB and 13.56% of serious delinquent UPB; the top segment 680-739 / 90-100 / 40-49 contributes 16.80% of portfolio delinquent UPB and 17.97% of serious delinquent UPB.,Risk management should focus on concentrated pockets of exposure rather than treating the portfolio as uniformly risky.
2,Credit score is the strongest descriptive risk separator,Observed delinquent and serious-delinquent exposure falls sharply as borrower credit quality improves.,credit_risk_ladder_pd / delinquency_driver_table / serious_delinquency_driver_table,"<620 delinquent UPB share = 7.98%; 620-679 = 6.10%; 680-739 = 2.23%; 740-799 = 0.48%; 800+ = 0.20%. Baseline-relative model odds ratio for active delinquency : <620 = 2.74, 620-679 = 1.99.","Credit quality should remain a primary dimension for underwriting, surveillance, pricing and capital allocation."
3,High LTV materially amplifies delinquency stress,"Leverage is a clear stress amplifier, especially in the 90-100 LTV band.",ltv_risk_ladder_pd / ltv_roll_rate_summary / delinquency_driver_table,"90-100 LTV delinquent UPB share = 1.91% versus 0.79% for <60 LTV. Baseline-relative model odds ratio for active delinquency, 90-100 LTV = 1.15.",High leverage reduces borrower resilience and should be closely monitored in portfolio surveillance and stress testing.
4,High DTI adds clear payment-stress risk,"Observed delinquent exposure rises meaningfully with debt burden, especially above 40%.",dti_risk_ladder_pd / dti_roll_rate_summary / delinquency_driver_table,"Delinquent UPB share rises from 0.44% for <20 DTI to 2.16% for 50+ DTI. Baseline-relative model odds ratio for active delinquency, 50+ DTI = 1.15.",DTI is a strong payment-stress signal and should be used alongside credit score and LTV in watchlist design.
5,"Risk is interaction-based, not single-factor","The most important risk pockets are combinations of moderate-to-weak credit, high LTV and elevated DTI.",segment_findings_pd / top_segment_risk_insight / credit_ltv_risk_summary,"The top flagged segment is 680-739 / 90-100 / 40-49; it contributes 16.80% of portfolio delinquent UPB and 17.97% of serious delinquent UPB, with a delinquent UPB risk index of 2.77 and a serious delinquent UPB risk index of 2.96.",Portfolio risk policy should target interacting borrower characteristics rather than relying on one-dimensional thresholds.
6,Distress persistence is strong in selected active-loan segments,"Among loans that remain active into the next monthly observation, several 60D-delinquent segments have high probabilities of rolling to 90D+ rather than curing.",distress_persistence_watchlist_pd / transition_30d_watchlist_pd,740-799 / 90-100 / 50+ rolls from 60D to 90D+ at 61.14%; 620-679 / 90-100 / 50+ rolls from 60D to 90D+ at 60.90%; 680-739 / 80-89 / 50+ rolls from 60D to 90D+ at 57.38%,"These segments are candidates for early intervention and elevated servicing attention, but the transition view should be read as active-to-active behavior rather than full-loan exit behavior."
7,Prepayment follows a different pattern from credit distress,"Higher-credit borrowers prepay earlier and at higher rates, especially in purchase and cash-out segments.",early_prepay_findings_pd / time_to_prepay_pd / prepay_driver_table,800+ / Purchase month-6 prepay share = 6.39%; baseline-relative model odds ratio for 800+ credit in month-6 prepayment = 1.41; conditional median time to first prepay is 6 months for 800+ / Purchase versus 9 months for 620-679 / Purchase.,"Safer borrowers may still create portfolio runoff and reinvestment risk, which matters to investors and servicing strategy."
8,Vintage differences are 